# 1.0 Import libraries

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import numpy as np
import pandas as pd
import altair as alt

# 2.0 Data import

In [5]:
url = "/content/drive/MyDrive/lmf/data/2026_08_31_web_visualizer/compiled_categories_quartiles_modified_for_CIAT-8719_and_CIAT-7714_to_recalculate_ranking.csv"
df = pd.read_csv(url)
df

,id_lab,id,subset,no,requisitioner,tax_name,functional_group,n_replicates_nutrition,dm_percentage,ash_dm,...,ch4_percentage_in_gas_8h,ch4_percentage_in_gas_24h,methane_intensity,tddm,ch4_category,tddm_category,lmf_category,lmf_category_rank,quartile,quartile_rank
0,F24-3470,CIAT-11194,1,199.0,Genetic_bank,Stylosanthes hamata,Herbaceous_legumes,2,92.89,9.61,...,15.17,18.02,46.12,59.25,Medium,Not High,Category_2,81.0,Q1,58.0
1,F24-3471,CIAT-11999,1,201.0,Genetic_bank,Stylosanthes guianensis,Herbaceous_legumes,2,93.50,10.46,...,12.98,16.25,46.61,58.55,Medium,Not High,Category_2,88.0,Q1,66.0
2,F24-3472,CIAT-12318,1,203.0,Genetic_bank,Stylosanthes hamata,Herbaceous_legumes,2,94.10,11.08,...,13.99,16.96,52.27,56.88,Medium,Not High,Category_2,148.0,Q2,78.0
3,F24-3427,CIAT-1257,1,113.0,Genetic_bank,Stylosanthes scabra,Herbaceous_legumes,2,92.72,9.52,...,15.71,17.60,46.32,58.95,Medium,Not High,Category_2,84.0,Q1,62.0
4,F24-3473,CIAT-13575,1,205.0,Genetic_bank,Desmodium incanum,Herbaceous_legumes,2,94.14,11.52,...,13.60,16.23,42.43,42.85,Low,Not High,Category_2,46.0,Q3,224.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
688,F25-2641,NaN,4,63.0,Mauricio Sotelo,Gliricidia sepium,Shrub_Trees,2,93.50,9.41,...,13.69,14.98,45.48,55.91,Medium,Not High,Category_2,151.0,NaN,NaN
689,F25-2641,NaN,4,63.0,Mauricio Sotelo,Gliricidia sepium,Shrub_Trees,2,93.50,9.41,...,13.69,14.98,45.48,55.91,Low,Not High,Category_2,1.0,NaN,NaN
690,F25-2641,NaN,4,63.0,Mauricio Sotelo,Gliricidia sepium,Shrub_Trees,2,93.50,9.41,...,13.69,14.98,45.48,55.91,High,Not High,Category_4,36.0,NaN,NaN
691,F25-2641,NaN,4,63.0,Mauricio Sotelo,Gliricidia sepium,Shrub_Trees,2,93.50,9.41,...,13.69,14.98,45.48,55.91,Low,Not High,Category_2,55.0,NaN,NaN


# 3.0 Summary, Counting banner

In [12]:
# ============================================================
# Requirements
# ============================================================
# pip install pandas
#
# NOTE: this piece is built with plain HTML/CSS (Flexbox), not
# Vega-Lite — the nested box layout (small boxes under big ones,
# plus a side box matching their combined height) is a layout
# problem, not a data-encoding one, and CSS handles it far more
# reliably than trying to force it into a chart grammar.
#
# This assumes `df` already exists, with "functional_group" and
# "lmf_category" columns.

import pandas as pd

points_all = df.copy()

valid_functional_groups = ["Grasses", "Herbaceous_legumes", "Shrub_Trees"]
points_all = points_all[points_all["functional_group"].isin(valid_functional_groups)].copy()

functional_group_order = ["Herbaceous_legumes", "Grasses", "Shrub_Trees"]
functional_group_colors = {
    "Herbaceous_legumes": "#008000",     # green
    "Grasses": "#6495ED",                # cornflowerblue
    "Shrub_Trees": "#FFA500",            # orange
}

# The 5 LMF categories shown as small boxes under each group.
# Adjust this list if your actual category set is different —
# any category not present in the data for a given group just
# shows 0.
CATEGORY_LABELS = ["Category_1a", "Category_1b", "Category_2", "Category_3", "Category_4"]
NEUTRAL_GRAY = "#B0B0B0"
TOTAL_BOX_COLOR = "purple"

# The overall sampling goal — the total box shows progress toward
# this number instead of a trivial "100%".
SAMPLING_GOAL = 2000


def blend(hex_color, target_hex, factor):
    """Blend hex_color toward target_hex by `factor` (0 = hex_color, 1 = target_hex)."""
    c = hex_color.lstrip("#")
    t = target_hex.lstrip("#")
    c_rgb = [int(c[i:i + 2], 16) for i in (0, 2, 4)]
    t_rgb = [int(t[i:i + 2], 16) for i in (0, 2, 4)]
    blended = [round(c_rgb[i] + (t_rgb[i] - c_rgb[i]) * factor) for i in range(3)]
    return "#{:02x}{:02x}{:02x}".format(*blended)


def shade_palette(base_hex, n):
    """n distinct shades of base_hex, from darker to lighter (base color in the middle)."""
    factors = [(-0.35, "#000000"), (-0.15, "#000000"), (0, None), (0.20, "#ffffff"), (0.40, "#ffffff")]
    shades = []
    for f, target in factors[:n]:
        if target is None:
            shades.append(base_hex)
        else:
            shades.append(blend(base_hex, target, abs(f)))
    return shades

# ============================================================
# Compute the numbers
# ============================================================

group_totals = points_all["functional_group"].value_counts().to_dict()

category_counts = {
    group: (
        points_all.loc[points_all["functional_group"] == group, "lmf_category"]
        .value_counts()
        .to_dict()
    )
    for group in functional_group_order
}

grand_total = len(points_all)
goal_percent = (grand_total / SAMPLING_GOAL * 100) if SAMPLING_GOAL else 0
goal_percent_str = f"{goal_percent:.0f}%"
goal_caption = f"of {SAMPLING_GOAL} goal"

# ============================================================
# Build the HTML
# ============================================================

def render_group_column(group):
    color = functional_group_colors[group]
    total = group_totals.get(group, 0)
    group_pct = (total / grand_total * 100) if grand_total else 0
    shades = shade_palette(color, len(CATEGORY_LABELS))

    sub_boxes_html = ""
    for cat, shade_color in zip(CATEGORY_LABELS, shades):
        count = category_counts.get(group, {}).get(cat, 0)
        cat_pct = (count / total * 100) if total else 0
        short_label = cat.replace("Category_", "")
        sub_boxes_html += f"""
        <div class="sub-box" style="background: {shade_color};">
          <div class="sub-box-label">{short_label}</div>
          <div class="sub-box-count">{count}</div>
          <div class="sub-box-percent">{cat_pct:.1f}%</div>
        </div>"""

    return f"""
    <div class="group-col">
      <div class="group-box" style="background: {color};">
        <div class="group-name">{group}</div>
        <div class="group-count">{total}</div>
        <div class="group-percent">{group_pct:.1f}%</div>
      </div>
      <div class="sub-boxes-row">
        {sub_boxes_html}
      </div>
    </div>"""


groups_html = "".join(render_group_column(g) for g in functional_group_order)

html_template = f"""
<!DOCTYPE html>
<html>
<head>
  <meta charset="utf-8" />
  <title>Functional Group Banner</title>
  <style>
    body {{
      font-family: sans-serif;
      display: flex;
      justify-content: center;
      padding: 30px;
    }}

    .dashboard {{
      display: flex;
      align-items: stretch;
      gap: 20px;
    }}

    .groups {{
      display: flex;
      gap: 16px;
    }}

    .total-box {{
      width: 220px;
      background: {TOTAL_BOX_COLOR};
      display: flex;
      flex-direction: column;
      align-items: center;
      justify-content: center;
      color: white;
      transition: background 0.2s ease;
    }}
    .total-label {{
      font-size: 15px;
      font-weight: bold;
      margin-bottom: 6px;
    }}
    .total-count {{
      font-size: 48px;
      font-weight: bold;
    }}
    .total-percent {{
      display: none;
      flex-direction: column;
      align-items: center;
      width: 100%;
    }}
    .total-percent-big {{
      font-size: 48px;
      font-weight: bold;
      text-align: center;
      width: 100%;
    }}
    .total-percent-caption {{
      font-size: 13px;
      font-weight: normal;
      opacity: 0.9;
      margin-top: 2px;
      text-align: center;
      width: 100%;
    }}
    .total-box:hover .total-count {{
      display: none;
    }}
    .total-box:hover .total-percent {{
      display: flex;
    }}
    /* Hovering ANY box (group, sub-box, or total) grays out every
       OTHER box in the whole dashboard — regardless of type or
       group. Only the exact box under the cursor keeps its color. */
    .dashboard:hover .group-box:not(:hover),
    .dashboard:hover .sub-box:not(:hover),
    .dashboard:hover .total-box:not(:hover) {{
      background: {NEUTRAL_GRAY} !important;
    }}

    .group-col {{
      display: flex;
      flex-direction: column;
      gap: 10px;
    }}

    .group-box {{
      width: 300px;
      height: 130px;
      border-radius: 0;
      display: flex;
      flex-direction: column;
      align-items: center;
      justify-content: center;
      color: white;
      transition: background 0.2s ease;
    }}
    /* (Cross-box graying handled by the single global rule above.) */
    .group-name {{
      font-size: 15px;
      font-weight: bold;
      margin-bottom: 6px;
    }}
    .group-count {{
      font-size: 40px;
      font-weight: bold;
    }}
    .group-percent {{
      font-size: 40px;
      font-weight: bold;
      display: none;
    }}
    .group-box:hover .group-count {{
      display: none;
    }}
    .group-box:hover .group-percent {{
      display: block;
    }}

    .sub-boxes-row {{
      display: flex;
      gap: 6px;
    }}

    .sub-box {{
      flex: 1;
      height: 70px;
      border-radius: 0;
      display: flex;
      flex-direction: column;
      align-items: center;
      justify-content: center;
      color: white;
    }}
    .sub-box-label {{
      font-size: 11px;
      font-weight: bold;
    }}
    .sub-box-count {{
      font-size: 18px;
      font-weight: bold;
    }}
    .sub-box-percent {{
      font-size: 15px;
      font-weight: bold;
      display: none;
    }}
    .sub-box:hover .sub-box-count {{
      display: none;
    }}
    .sub-box:hover .sub-box-percent {{
      display: block;
    }}
  </style>
</head>
<body>

  <div class="dashboard">
    <div class="total-box">
      <div class="total-label">Total sampled</div>
      <div class="total-count">{grand_total}</div>
      <div class="total-percent">
        <div class="total-percent-big">{goal_percent_str}</div>
        <div class="total-percent-caption">{goal_caption}</div>
      </div>
    </div>

    <div class="groups">
      {groups_html}
    </div>
  </div>

</body>
</html>
"""

with open("banner.html", "w", encoding="utf-8") as f:
    f.write(html_template)

print("Done: banner.html")

Done: banner.html


In [14]:
with open("/content/drive/MyDrive/lmf/output/2026_08_31_web_visualizer/banner.html", "w", encoding="utf-8") as f:
    f.write(html_template)

# 4.0 Data composition, Pie and Bar charts

## 4.1 Pie chart - Functional groups

In [15]:
# ============================================================
# Requirements
# ============================================================
# pip install altair vl-convert-python pandas numpy
#
# vl-convert-python is required so that chart.to_json()-based
# rendering works cleanly.

import re
import json
import pandas as pd
import altair as alt

# Disable Altair's default 5000-row safety limit, in case df is large.
alt.data_transformers.disable_max_rows()

# ============================================================
# NOTE: this assumes `df` already exists, with at least the columns
# "functional_group", "subset", "lmf_category", "quartile", and
# "id_lab" (formatted like "F24-0001", where "24" -> year 2024).
# ============================================================

points_all = df.copy()

# Only these three functional groups are relevant here — same
# restriction used in the scatterplot.
valid_functional_groups = ["Grasses", "Herbaceous_legumes", "Shrub_Trees"]
points_all = points_all[points_all["functional_group"].isin(valid_functional_groups)].copy()

# Same color scheme used across PCA / UMAP / scatterplot, for
# visual consistency across the whole toolkit.
functional_group_order = ["Herbaceous_legumes", "Grasses", "Shrub_Trees"]
functional_group_colors = ["green", "cornflowerblue", "orange"]

# ============================================================
# Processing year — extracted from id_lab's 2-digit code right
# before the dash (e.g. "F24-0001" -> "24" -> 2024). Used here only
# to build the dropdown's list of options in Python; the actual
# live filtering re-derives the same year from id_lab directly in
# the browser (see year_calc_expr below), so it works regardless of
# which rows are currently visible.
# ============================================================


def extract_year(id_lab):
    match = re.search(r"(\d{2})-", str(id_lab))
    return f"20{match.group(1)}" if match else None


points_all["_year"] = points_all["id_lab"].apply(extract_year)
year_options = ["All"] + sorted(points_all["_year"].dropna().unique().tolist())

# Vega expression that reproduces the same extraction logic
# (2 digits immediately before the first "-") directly on id_lab,
# so it's recalculated live for whatever rows are in view.
year_calc_expr = (
    "'20' + slice(datum.id_lab, indexof(datum.id_lab, '-') - 2, indexof(datum.id_lab, '-'))"
)

# ============================================================
# Dropdown options for subset / category / year
# ============================================================

subset_options = ["All"] + sorted(points_all["subset"].astype(str).unique().tolist())
category_options = ["All"] + sorted(points_all["lmf_category"].dropna().unique().tolist())

# ============================================================
# Interactive controls
# ============================================================

sel_subset = alt.param(
    name="sel_subset",
    value="All",
    bind=alt.binding_select(options=subset_options, name="Subset: ")
)

sel_category = alt.param(
    name="sel_category",
    value="All",
    bind=alt.binding_select(options=category_options, name="Category: ")
)

sel_year = alt.param(
    name="sel_year",
    value="All",
    bind=alt.binding_select(options=year_options, name="Processing year: ")
)

filter_expr = (
    "(sel_subset == 'All' || toString(datum.subset) == sel_subset) && "
    "(sel_category == 'All' || datum.lmf_category == sel_category) && "
    f"(sel_year == 'All' || ({year_calc_expr}) == sel_year)"
)

# ============================================================
# Title row — shows the current filter selection and total n,
# recalculated live in the browser via aggregation (no precomputed
# combinations needed, since this is just a row count).
# ============================================================

title_chart = (
    alt.Chart(points_all)
    .transform_filter(filter_expr)
    .transform_aggregate(n="count()")
    .transform_calculate(
        label=(
            "'Functional Group Composition'"
            " + (sel_subset == 'All' ? '' : '  |  Subset: ' + sel_subset)"
            " + (sel_category == 'All' ? '' : '  |  Category: ' + sel_category)"
            " + (sel_year == 'All' ? '' : '  |  Year: ' + sel_year)"
        )
    )
    .mark_text(align="center", fontSize=18, fontWeight="bold", dy=0)
    .encode(text="label:N")
    .properties(width=500, height=30)
)

# ============================================================
# Pie chart — percentage of samples per functional group,
# recalculated live from whatever subset/category/year is
# currently selected.
# ============================================================

pie_base = (
    alt.Chart(points_all)
    .transform_filter(filter_expr)
    .transform_aggregate(count="count()", groupby=["functional_group"])
    .transform_joinaggregate(total="sum(count)")
    .transform_calculate(percentage="datum.count / datum.total * 100")
    .transform_calculate(label_top="format(datum.percentage, '.1f') + '%'")
    .transform_calculate(label_bottom="'N = ' + datum.count")
    # Cumulative sum (in the same descending-by-count order used
    # visually) computed BEFORE any hover filtering, so each slice's
    # angular position is fixed and independent of which rows remain
    # visible afterward.
    .transform_window(
        cum_count="sum(count)",
        sort=[
            {"field": "count", "order": "descending"},
            {"field": "functional_group", "order": "ascending"}
        ],
        frame=[None, 0]
    )
    .transform_calculate(theta_start="(datum.cum_count - datum.count) / datum.total * 2 * 3.14159265358979")
    .transform_calculate(theta_end="datum.cum_count / datum.total * 2 * 3.14159265358979")
    .transform_calculate(theta_mid="(datum.theta_start + datum.theta_end) / 2")
)

FULL_CIRCLE_SCALE = alt.Scale(domain=[0, 2 * 3.14159265358979])

hover = alt.selection_point(
    fields=["functional_group"],
    on="pointerover",
    clear="pointerout",
    empty="all",
    name="hover"
)

base_slices = (
    pie_base
    .mark_arc(
        innerRadius=145,
        outerRadius=205,
        padAngle=0.015,
        cornerRadius=0,
        stroke="white",
        strokeWidth=2
    )
    .encode(
        theta=alt.Theta("theta_start:Q", scale=FULL_CIRCLE_SCALE, title=None),
        theta2=alt.Theta2("theta_end:Q"),
        color=alt.Color(
            "functional_group:N",
            scale=alt.Scale(domain=functional_group_order, range=functional_group_colors),
            legend=alt.Legend(title="Functional Group", symbolSize=100)
        ),
        opacity=alt.condition(hover, alt.value(1), alt.value(0.35)),
        tooltip=[
            alt.Tooltip("functional_group:N", title="Functional group"),
            alt.Tooltip("count:Q", title="N samples"),
            alt.Tooltip("percentage:Q", title="Percentage", format=".1f")
        ]
    )
    .add_params(hover)
)

label_top_layer = (
    pie_base
    .mark_text(radius=240, dy=-8, fontSize=16, fontWeight="bold")
    .encode(
        theta=alt.Theta("theta_mid:Q", scale=FULL_CIRCLE_SCALE, title=None),
        text=alt.Text("label_top:N"),
        color=alt.value("#333333")
    )
)

label_bottom_layer = (
    pie_base
    .mark_text(radius=240, dy=12, fontSize=12)
    .encode(
        theta=alt.Theta("theta_mid:Q", scale=FULL_CIRCLE_SCALE, title=None),
        text=alt.Text("label_bottom:N"),
        color=alt.value("#666666")
    )
)

# Center label inside the donut hole, showing the overall total.
center_total = (
    alt.Chart(points_all)
    .transform_filter(filter_expr)
    .transform_aggregate(n="count()")
    .transform_calculate(center_label="'Total\\nN = ' + datum.n")
    .mark_text(fontSize=20, fontWeight="bold", color="#333333", lineBreak="\\n", align="center")
    .encode(text="center_label:N")
)

pie_chart = (
    (base_slices + label_top_layer + label_bottom_layer + center_total)
    .properties(width=520, height=520)
)

# ============================================================
# Final layout — controls above, then title, then the chart
# ============================================================

final_plot = (
    alt.vconcat(title_chart, pie_chart)
    .add_params(sel_subset, sel_category, sel_year)
    .configure_legend(titleFontSize=16, labelFontSize=14)
    .configure_view(strokeWidth=0)
)

# ============================================================
# Save as interactive HTML with controls forced above the chart
# ============================================================

chart_spec = final_plot.to_json(indent=None)

html_template = f"""
<!DOCTYPE html>
<html>
<head>
  <meta charset="utf-8" />
  <title>Functional Group Pie Chart</title>
  <script src="https://cdn.jsdelivr.net/npm/vega@5"></script>
  <script src="https://cdn.jsdelivr.net/npm/vega-lite@5"></script>
  <script src="https://cdn.jsdelivr.net/npm/vega-embed@6"></script>
  <script src="https://cdn.jsdelivr.net/npm/utif@3.1.0/UTIF.js"></script>
  <script src="https://cdnjs.cloudflare.com/ajax/libs/jspdf/2.5.1/jspdf.umd.min.js"></script>
  <style>
    body {{
      font-family: sans-serif;
      display: flex;
      flex-direction: column;
      align-items: center;
    }}
    #controls {{
      margin: 20px 0;
      font-size: 16px;
    }}
    #download-buttons {{
      margin: 20px 0 10px 0;
      display: flex;
      align-items: center;
      gap: 10px;
    }}
    #download-buttons select,
    #download-buttons button {{
      padding: 8px 16px;
      font-size: 14px;
      border: 1px solid #ccc;
      border-radius: 6px;
      background: #f7f7f9;
      cursor: pointer;
    }}
    #download-buttons button:hover {{
      background: #e9e9ee;
    }}
    /* Smooth hover animation for the donut slices */
    #vis path {{
      transition: opacity 0.2s ease;
    }}
  </style>
</head>
<body>
  <div id="controls"></div>
  <div id="vis"></div>
  <div id="download-buttons">
    <select id="format-select">
      <option value="png">PNG</option>
      <option value="jpg">JPG</option>
      <option value="tiff">TIFF</option>
      <option value="pdf">PDF</option>
    </select>
    <button id="btn-download">Download</button>
  </div>

  <script type="text/javascript">
    const spec = {chart_spec};

    const DPI = 300;
    const DPI_SCALE = DPI / 96;

    vegaEmbed("#vis", spec, {{
      actions: false,
      bindingsElement: "#controls",
      renderer: "svg"
    }}).then(function(result) {{

      const view = result.view;

      function triggerDownload(url, filename) {{
        const link = document.createElement("a");
        link.href = url;
        link.download = filename;
        document.body.appendChild(link);
        link.click();
        document.body.removeChild(link);
      }}

      function downloadAs(format) {{
        view.toCanvas(DPI_SCALE).then(function(canvas) {{

          if (format === "png") {{
            triggerDownload(canvas.toDataURL("image/png"), "functional_group_pie.png");

          }} else if (format === "jpg") {{
            triggerDownload(canvas.toDataURL("image/jpeg", 0.95), "functional_group_pie.jpg");

          }} else if (format === "tiff") {{
            const ctx = canvas.getContext("2d");
            const imageData = ctx.getImageData(0, 0, canvas.width, canvas.height);
            const tiffBuffer = UTIF.encodeImage(imageData.data, canvas.width, canvas.height);
            const blob = new Blob([tiffBuffer], {{ type: "image/tiff" }});
            const url = URL.createObjectURL(blob);
            triggerDownload(url, "functional_group_pie.tiff");

          }} else if (format === "pdf") {{
            const widthIn = canvas.width / DPI;
            const heightIn = canvas.height / DPI;
            const {{ jsPDF }} = window.jspdf;
            const pdf = new jsPDF({{
              orientation: widthIn >= heightIn ? "landscape" : "portrait",
              unit: "in",
              format: [widthIn, heightIn]
            }});
            pdf.addImage(canvas.toDataURL("image/png"), "PNG", 0, 0, widthIn, heightIn);
            pdf.save("functional_group_pie.pdf");
          }}

        }}).catch(console.error);
      }}

      document.getElementById("btn-download").addEventListener("click", function() {{
        const format = document.getElementById("format-select").value;
        downloadAs(format);
      }});

    }}).catch(console.error);
  </script>
</body>
</html>
"""

with open("pie_chart.html", "w", encoding="utf-8") as f:
    f.write(html_template)

print("Done: pie_chart.html")

Done: pie_chart.html


In [16]:
with open("/content/drive/MyDrive/lmf/output/2026_08_31_web_visualizer/pie_chart.html", "w", encoding="utf-8") as f:
    f.write(html_template)

## 4.2 Bar chart - Species

In [17]:
# ============================================================
# Requirements
# ============================================================
# pip install altair vl-convert-python pandas numpy
#
# vl-convert-python is required so that chart.to_json()-based
# rendering works cleanly.

import re
import pandas as pd
import altair as alt

alt.data_transformers.disable_max_rows()

# ============================================================
# NOTE: this assumes `df` already exists, with at least the columns
# "tax_name", "subset", "lmf_category", and "id_lab" (formatted
# like "F24-0001", where "24" -> year 2024).
# ============================================================

points_all = df.copy()
points_all = points_all[points_all["tax_name"].notna()].copy()

# Only these three functional groups are relevant — same restriction
# used everywhere else in the toolkit.
valid_functional_groups = ["Grasses", "Herbaceous_legumes", "Shrub_Trees"]
points_all = points_all[points_all["functional_group"].isin(valid_functional_groups)].copy()

# How many individual species to show before collapsing the rest
# into a single "Other species" slice. Tune this to taste.
TOP_N_SPECIES = 8

# ============================================================
# Processing year (same extraction logic as the functional-group
# pie) — "F24-0001" -> "2024".
# ============================================================


def extract_year(id_lab):
    match = re.search(r"(\d{2})-", str(id_lab))
    return f"20{match.group(1)}" if match else None


points_all["_year"] = points_all["id_lab"].apply(extract_year)
year_options = ["All"] + sorted(points_all["_year"].dropna().unique().tolist())

year_calc_expr = (
    "'20' + slice(datum.id_lab, indexof(datum.id_lab, '-') - 2, indexof(datum.id_lab, '-'))"
)

# ============================================================
# Dropdown options
# ============================================================

subset_options = ["All"] + sorted(points_all["subset"].astype(str).unique().tolist())
category_options = ["All"] + sorted(points_all["lmf_category"].dropna().unique().tolist())
functional_group_options = ["All"] + sorted(points_all["functional_group"].dropna().unique().tolist())

# Species dropdown: shows "Species name (N accessions)" in the list,
# but the underlying value used for filtering is just the species
# name. Counts are computed once from the full (3-group-restricted)
# dataset, so they reflect overall totals rather than whatever other
# filters happen to be active at any given moment.
tax_name_counts = points_all["tax_name"].value_counts().to_dict()
tax_name_order = sorted(points_all["tax_name"].dropna().unique().tolist())
tax_name_select_values = ["All"] + tax_name_order
tax_name_select_labels = ["All"] + [
    f"{name} ({tax_name_counts.get(name, 0)})" for name in tax_name_order
]

# ============================================================
# Interactive controls
# ============================================================

sel_subset = alt.param(
    name="sel_subset", value="All",
    bind=alt.binding_select(options=subset_options, name="Subset: ")
)
sel_category = alt.param(
    name="sel_category", value="All",
    bind=alt.binding_select(options=category_options, name="Category: ")
)
sel_group = alt.param(
    name="sel_group", value="All",
    bind=alt.binding_select(options=functional_group_options, name="Functional group: ")
)
sel_year = alt.param(
    name="sel_year", value="All",
    bind=alt.binding_select(options=year_options, name="Processing year: ")
)
sel_tax_name = alt.param(
    name="sel_tax_name", value="All",
    bind=alt.binding_select(
        options=tax_name_select_values,
        labels=tax_name_select_labels,
        name="Species: "
    )
)

filter_expr = (
    "(sel_subset == 'All' || toString(datum.subset) == sel_subset) && "
    "(sel_category == 'All' || datum.lmf_category == sel_category) && "
    "(sel_group == 'All' || datum.functional_group == sel_group) && "
    "(sel_tax_name == 'All' || datum.tax_name == sel_tax_name) && "
    f"(sel_year == 'All' || ({year_calc_expr}) == sel_year)"
)

# ============================================================
# Title — current filter selection + a live species/accession count
# ============================================================

title_chart = (
    alt.Chart(points_all)
    .transform_filter(filter_expr)
    .transform_aggregate(n_accessions="count()", n_species="distinct(tax_name)")
    .transform_calculate(
        label=(
            "'Species Composition'"
            " + (sel_group == 'All' ? '' : '  |  Group: ' + sel_group)"
            " + (sel_tax_name == 'All' ? '' : '  |  Species: ' + sel_tax_name)"
            " + (sel_subset == 'All' ? '' : '  |  Subset: ' + sel_subset)"
            " + (sel_category == 'All' ? '' : '  |  Category: ' + sel_category)"
            " + (sel_year == 'All' ? '' : '  |  Year: ' + sel_year)"
        )
    )
    .mark_text(align="center", fontSize=18, fontWeight="bold", dy=0)
    .encode(text="label:N")
    .properties(width=520, height=30)
)

# ============================================================
# Pie chart — accessions per species, top N individually shown,
# the rest collapsed into "Other species". All recalculated live
# from whatever filters are currently active.
# ============================================================

pie_base = (
    alt.Chart(points_all)
    .transform_filter(filter_expr)
    # Step 1: accessions per species
    .transform_aggregate(species_count="count()", groupby=["tax_name"])
    # Step 2: give each species a strictly unique sequential number,
    # ordered by accession count (highest first). Using row_number()
    # instead of rank() matters here: rank() gives EVERY tied species
    # (e.g. several species that all have exactly 1 accession) the
    # SAME rank number, so a whole tied group could all slip under
    # the TOP_N threshold at once instead of being collapsed into
    # "Other species" — that was producing dozens of near-invisible
    # slivers instead of one clean "Other" slice, which is what was
    # showing up as gaps in the donut.
    .transform_window(
        row_number="row_number()",
        sort=[
            {"field": "species_count", "order": "descending"},
            {"field": "tax_name", "order": "ascending"}
        ]
    )
    # Step 3: keep the top N species by name, collapse the rest
    .transform_calculate(
        species_group=f"datum.row_number <= {TOP_N_SPECIES} ? datum.tax_name : 'Other species'"
    )
    # Step 4: re-aggregate by that bucketed label — sum(species_count)
    # gives the accession total per slice, count() gives how many
    # distinct species got folded into that slice (relevant for
    # "Other species").
    .transform_aggregate(
        count="sum(species_count)",
        n_species_in_slice="count()",
        groupby=["species_group"]
    )
    .transform_joinaggregate(total="sum(count)")
    .transform_calculate(percentage="datum.count / datum.total * 100")
    .transform_calculate(label_top="format(datum.percentage, '.1f') + '%'")
    .transform_calculate(label_bottom="'N = ' + datum.count")
    .transform_window(
        cum_count="sum(count)",
        sort=[
            {"field": "count", "order": "descending"},
            {"field": "species_group", "order": "ascending"}
        ],
        frame=[None, 0]
    )
    .transform_calculate(theta_start="(datum.cum_count - datum.count) / datum.total * 2 * 3.14159265358979")
    .transform_calculate(theta_end="datum.cum_count / datum.total * 2 * 3.14159265358979")
    .transform_calculate(theta_mid="(datum.theta_start + datum.theta_end) / 2")
)

FULL_CIRCLE_SCALE = alt.Scale(domain=[0, 2 * 3.14159265358979])

hover = alt.selection_point(
    fields=["species_group"],
    on="pointerover",
    clear="pointerout",
    empty="all",
    name="hover"
)

base_slices = (
    pie_base
    .mark_arc(
        innerRadius=145,
        outerRadius=205,
        padAngle=0.006,
        cornerRadius=0,
        stroke="white",
        strokeWidth=1
    )
    .encode(
        theta=alt.Theta("theta_start:Q", scale=FULL_CIRCLE_SCALE, title=None),
        theta2=alt.Theta2("theta_end:Q"),
        color=alt.Color(
            "species_group:N",
            sort=alt.SortField("count", order="descending"),
            scale=alt.Scale(scheme="tableau20"),
            legend=alt.Legend(title="Species (top {} + other)".format(TOP_N_SPECIES), symbolSize=90, labelLimit=220)
        ),
        opacity=alt.condition(hover, alt.value(1), alt.value(0.35)),
        tooltip=[
            alt.Tooltip("species_group:N", title="Species"),
            alt.Tooltip("count:Q", title="N accessions"),
            alt.Tooltip("percentage:Q", title="Percentage", format=".1f"),
            alt.Tooltip("n_species_in_slice:Q", title="Species in this slice")
        ]
    )
    .add_params(hover)
)

label_top_layer = (
    pie_base
    .mark_text(radius=240, dy=-8, fontSize=15, fontWeight="bold")
    .encode(
        theta=alt.Theta("theta_mid:Q", scale=FULL_CIRCLE_SCALE, title=None),
        text=alt.Text("label_top:N"),
        color=alt.value("#333333")
    )
)

label_bottom_layer = (
    pie_base
    .mark_text(radius=240, dy=12, fontSize=11)
    .encode(
        theta=alt.Theta("theta_mid:Q", scale=FULL_CIRCLE_SCALE, title=None),
        text=alt.Text("label_bottom:N"),
        color=alt.value("#666666")
    )
)

# Center label — total distinct species AND total accessions,
# always reflecting the current filters (not the top-N grouping).
center_total = (
    alt.Chart(points_all)
    .transform_filter(filter_expr)
    .transform_aggregate(n_accessions="count()", n_species="distinct(tax_name)")
    .transform_calculate(
        center_label="datum.n_species + ' species\\n' + datum.n_accessions + ' accessions'"
    )
    .mark_text(fontSize=18, fontWeight="bold", color="#333333", lineBreak="\\n", align="center")
    .encode(text="center_label:N")
)

pie_chart = (
    (base_slices + label_top_layer + label_bottom_layer + center_total)
    .properties(width=520, height=520)
)

# ============================================================
# Final layout
# ============================================================

final_plot = (
    alt.vconcat(title_chart, pie_chart)
    .add_params(sel_subset, sel_category, sel_group, sel_tax_name, sel_year)
    .configure_legend(titleFontSize=15, labelFontSize=12)
    .configure_view(strokeWidth=0)
)

# ============================================================
# Save as interactive HTML with controls forced above the chart
# ============================================================

chart_spec = final_plot.to_json(indent=None)

html_template = f"""
<!DOCTYPE html>
<html>
<head>
  <meta charset="utf-8" />
  <title>Species Pie Chart</title>
  <script src="https://cdn.jsdelivr.net/npm/vega@5"></script>
  <script src="https://cdn.jsdelivr.net/npm/vega-lite@5"></script>
  <script src="https://cdn.jsdelivr.net/npm/vega-embed@6"></script>
  <script src="https://cdn.jsdelivr.net/npm/utif@3.1.0/UTIF.js"></script>
  <script src="https://cdnjs.cloudflare.com/ajax/libs/jspdf/2.5.1/jspdf.umd.min.js"></script>
  <style>
    body {{
      font-family: sans-serif;
      display: flex;
      flex-direction: column;
      align-items: center;
    }}
    #controls {{
      margin: 20px 0;
      font-size: 16px;
    }}
    #download-buttons {{
      margin: 20px 0 10px 0;
      display: flex;
      align-items: center;
      gap: 10px;
    }}
    #download-buttons select,
    #download-buttons button {{
      padding: 8px 16px;
      font-size: 14px;
      border: 1px solid #ccc;
      border-radius: 6px;
      background: #f7f7f9;
      cursor: pointer;
    }}
    #download-buttons button:hover {{
      background: #e9e9ee;
    }}
    #vis path {{
      transition: opacity 0.2s ease;
    }}
  </style>
</head>
<body>
  <div id="controls"></div>
  <div id="vis"></div>
  <div id="download-buttons">
    <select id="format-select">
      <option value="png">PNG</option>
      <option value="jpg">JPG</option>
      <option value="tiff">TIFF</option>
      <option value="pdf">PDF</option>
    </select>
    <button id="btn-download">Download</button>
  </div>

  <script type="text/javascript">
    const spec = {chart_spec};

    const DPI = 300;
    const DPI_SCALE = DPI / 96;

    vegaEmbed("#vis", spec, {{
      actions: false,
      bindingsElement: "#controls",
      renderer: "svg"
    }}).then(function(result) {{

      const view = result.view;

      function triggerDownload(url, filename) {{
        const link = document.createElement("a");
        link.href = url;
        link.download = filename;
        document.body.appendChild(link);
        link.click();
        document.body.removeChild(link);
      }}

      function downloadAs(format) {{
        view.toCanvas(DPI_SCALE).then(function(canvas) {{

          if (format === "png") {{
            triggerDownload(canvas.toDataURL("image/png"), "species_pie.png");

          }} else if (format === "jpg") {{
            triggerDownload(canvas.toDataURL("image/jpeg", 0.95), "species_pie.jpg");

          }} else if (format === "tiff") {{
            const ctx = canvas.getContext("2d");
            const imageData = ctx.getImageData(0, 0, canvas.width, canvas.height);
            const tiffBuffer = UTIF.encodeImage(imageData.data, canvas.width, canvas.height);
            const blob = new Blob([tiffBuffer], {{ type: "image/tiff" }});
            const url = URL.createObjectURL(blob);
            triggerDownload(url, "species_pie.tiff");

          }} else if (format === "pdf") {{
            const widthIn = canvas.width / DPI;
            const heightIn = canvas.height / DPI;
            const {{ jsPDF }} = window.jspdf;
            const pdf = new jsPDF({{
              orientation: widthIn >= heightIn ? "landscape" : "portrait",
              unit: "in",
              format: [widthIn, heightIn]
            }});
            pdf.addImage(canvas.toDataURL("image/png"), "PNG", 0, 0, widthIn, heightIn);
            pdf.save("species_pie.pdf");
          }}

        }}).catch(console.error);
      }}

      document.getElementById("btn-download").addEventListener("click", function() {{
        const format = document.getElementById("format-select").value;
        downloadAs(format);
      }});

    }}).catch(console.error);
  </script>
</body>
</html>
"""

with open("species_pie.html", "w", encoding="utf-8") as f:
    f.write(html_template)

print("Done: species_pie.html")

Done: species_pie.html


In [18]:
with open("/content/drive/MyDrive/lmf/output/2026_08_31_web_visualizer/species_pie.html", "w", encoding="utf-8") as f:
    f.write(html_template)

## 4.3 Bar chart - Categories

In [19]:
# ============================================================
# Requirements
# ============================================================
# pip install altair vl-convert-python pandas numpy
#
# vl-convert-python is required so that chart.to_json()-based
# rendering works cleanly.

import pandas as pd
import altair as alt

alt.data_transformers.disable_max_rows()

# ============================================================
# NOTE: this assumes `df` already exists, with at least the columns
# "functional_group" and "lmf_category" (same data used for the pie).
# ============================================================

points_all = df.copy()

valid_functional_groups = ["Grasses", "Herbaceous_legumes", "Shrub_Trees"]
points_all = points_all[points_all["functional_group"].isin(valid_functional_groups)].copy()

functional_group_order = ["Grasses", "Herbaceous_legumes", "Shrub_Trees"]

# Categories are read directly from the data, so this adapts to
# whatever LMF categories your real dataset actually contains.
found_categories = sorted(points_all["lmf_category"].dropna().unique().tolist())

# Taxon names — dropdown shows "Species name (N accessions)", but
# the underlying filter value is just the species name. Counts are
# computed once from the full (3-group-restricted) dataset.
tax_name_counts = points_all["tax_name"].value_counts().to_dict()
tax_name_order = sorted(points_all["tax_name"].dropna().unique().tolist())
tax_name_select_values = ["All"] + tax_name_order
tax_name_select_labels = ["All"] + [
    f"{name} ({tax_name_counts.get(name, 0)})" for name in tax_name_order
]

# Fixed, explicit color per category NAME (matching the reference
# image) — intentionally NOT positional/index-based, so a category
# being entirely absent from the data (or from a specific functional
# group / filter selection) never shifts the color of any other
# category.
CATEGORY_COLOR_MAP = {
    "Category_1a": "#1CADA8",   # teal
    "Category_1b": "#8BC63F",   # green
    "Category_2": "#E8A93B",    # gold
    "Category_3": "#A9CC93",    # light green
    "Category_4": "#E6E6E6",    # gray
}
FALLBACK_PALETTE = ["#B07AA1", "#E15759", "#76B7B2", "#FF9DA7", "#9C755F"]

# Preserve the canonical order first (only the categories actually
# present), then append any unexpected category names not in the
# fixed map, giving those a fallback color.
category_order = (
    [c for c in CATEGORY_COLOR_MAP if c in found_categories]
    + [c for c in found_categories if c not in CATEGORY_COLOR_MAP]
)

category_colors = {}
_fallback_i = 0
for c in category_order:
    if c in CATEGORY_COLOR_MAP:
        category_colors[c] = CATEGORY_COLOR_MAP[c]
    else:
        category_colors[c] = FALLBACK_PALETTE[_fallback_i % len(FALLBACK_PALETTE)]
        _fallback_i += 1

# ============================================================
# Dropdown filters — same style as the rest of the toolkit
# (single-select "All" dropdowns, not checkboxes).
# ============================================================

group_options = ["All"] + functional_group_order
category_options = ["All"] + category_order

sel_group = alt.param(
    name="sel_group",
    value="All",
    bind=alt.binding_select(options=group_options, name="Functional group: ")
)

sel_category = alt.param(
    name="sel_category",
    value="All",
    bind=alt.binding_select(options=category_options, name="Category: ")
)

sel_tax = alt.param(
    name="sel_tax",
    value="All",
    bind=alt.binding_select(
        options=tax_name_select_values,
        labels=tax_name_select_labels,
        name="Taxon name: "
    )
)

filter_expr = (
    "(sel_group == 'All' || datum.functional_group == sel_group) && "
    "(sel_category == 'All' || datum.lmf_category == sel_category) && "
    "(sel_tax == 'All' || datum.tax_name == sel_tax)"
)

# ============================================================
# Title
# ============================================================

title_chart = (
    alt.Chart(pd.DataFrame({"_dummy": [1]}))
    .transform_calculate(
        label=(
            "sel_group == 'All' "
            "? 'Number of accessions by Functional Group' "
            ": 'Number of accessions by Category'"
        )
    )
    .mark_text(align="center", fontSize=18, fontWeight="bold", dy=0)
    .encode(text="label:N")
    .properties(width=550, height=30)
)

# ============================================================
# Grouped bar chart — no legend; each bar carries its own
# category label + count directly above it.
# ============================================================

bar_base = (
    alt.Chart(points_all)
    .transform_filter(filter_expr)
    .transform_aggregate(count="count()", groupby=["functional_group", "lmf_category"])
    .transform_calculate(short_cat="replace(datum.lmf_category, 'Category_', '')")
    .transform_calculate(bar_label="datum.short_cat + ': ' + datum.count")
)

bars = (
    bar_base
    .mark_bar()
    .encode(
        x=alt.X(
            "functional_group:N",
            sort=functional_group_order,
            title="Functional Group",
            axis=alt.Axis(labelAngle=0, labelPadding=8)
        ),
        xOffset=alt.XOffset(
            "lmf_category:N",
            sort=category_order,
            scale=alt.Scale(paddingInner=0.15)
        ),
        y=alt.Y("count:Q", title="Number of accessions", axis=alt.Axis(grid=False)),
        color=alt.Color(
            "lmf_category:N",
            sort=category_order,
            scale=alt.Scale(domain=category_order, range=[category_colors[c] for c in category_order]),
            legend=None
        ),
        tooltip=[
            alt.Tooltip("functional_group:N", title="Functional group"),
            alt.Tooltip("lmf_category:N", title="LMF category"),
            alt.Tooltip("count:Q", title="N accessions")
        ]
    )
)

bar_chart = bars.properties(width=550, height=420)

# ============================================================
# Final layout
# ============================================================

final_plot = (
    alt.vconcat(title_chart, bar_chart)
    .add_params(sel_group, sel_category, sel_tax)
    .configure_axis(labelFontSize=13, titleFontSize=15)
    .configure_view(strokeWidth=0)
)

# ============================================================
# Save as interactive HTML with the dropdowns above the chart
# ============================================================

chart_spec = final_plot.to_json(indent=None)

html_template = f"""
<!DOCTYPE html>
<html>
<head>
  <meta charset="utf-8" />
  <title>Priority Observations by Group and Category</title>
  <script src="https://cdn.jsdelivr.net/npm/vega@5"></script>
  <script src="https://cdn.jsdelivr.net/npm/vega-lite@5"></script>
  <script src="https://cdn.jsdelivr.net/npm/vega-embed@6"></script>
  <script src="https://cdn.jsdelivr.net/npm/utif@3.1.0/UTIF.js"></script>
  <script src="https://cdnjs.cloudflare.com/ajax/libs/jspdf/2.5.1/jspdf.umd.min.js"></script>
  <style>
    body {{
      font-family: sans-serif;
      display: flex;
      flex-direction: column;
      align-items: center;
    }}
    #controls {{
      margin: 20px 0;
      font-size: 16px;
    }}
    #download-buttons {{
      margin: 20px 0 10px 0;
      display: flex;
      align-items: center;
      gap: 10px;
    }}
    #download-buttons select,
    #download-buttons button {{
      padding: 8px 16px;
      font-size: 14px;
      border: 1px solid #ccc;
      border-radius: 6px;
      background: #f7f7f9;
      cursor: pointer;
    }}
    #download-buttons button:hover {{
      background: #e9e9ee;
    }}
  </style>
</head>
<body>
  <div id="controls"></div>
  <div id="vis"></div>
  <div id="download-buttons">
    <select id="format-select">
      <option value="png">PNG</option>
      <option value="jpg">JPG</option>
      <option value="tiff">TIFF</option>
      <option value="pdf">PDF</option>
    </select>
    <button id="btn-download">Download</button>
  </div>

  <script type="text/javascript">
    const spec = {chart_spec};

    const DPI = 300;
    const DPI_SCALE = DPI / 96;

    vegaEmbed("#vis", spec, {{
      actions: false,
      bindingsElement: "#controls",
      renderer: "svg"
    }}).then(function(result) {{

      const view = result.view;

      function triggerDownload(url, filename) {{
        const link = document.createElement("a");
        link.href = url;
        link.download = filename;
        document.body.appendChild(link);
        link.click();
        document.body.removeChild(link);
      }}

      function downloadAs(format) {{
        view.toCanvas(DPI_SCALE).then(function(canvas) {{

          if (format === "png") {{
            triggerDownload(canvas.toDataURL("image/png"), "priority_observations.png");

          }} else if (format === "jpg") {{
            triggerDownload(canvas.toDataURL("image/jpeg", 0.95), "priority_observations.jpg");

          }} else if (format === "tiff") {{
            const ctx = canvas.getContext("2d");
            const imageData = ctx.getImageData(0, 0, canvas.width, canvas.height);
            const tiffBuffer = UTIF.encodeImage(imageData.data, canvas.width, canvas.height);
            const blob = new Blob([tiffBuffer], {{ type: "image/tiff" }});
            const url = URL.createObjectURL(blob);
            triggerDownload(url, "priority_observations.tiff");

          }} else if (format === "pdf") {{
            const widthIn = canvas.width / DPI;
            const heightIn = canvas.height / DPI;
            const {{ jsPDF }} = window.jspdf;
            const pdf = new jsPDF({{
              orientation: widthIn >= heightIn ? "landscape" : "portrait",
              unit: "in",
              format: [widthIn, heightIn]
            }});
            pdf.addImage(canvas.toDataURL("image/png"), "PNG", 0, 0, widthIn, heightIn);
            pdf.save("priority_observations.pdf");
          }}

        }}).catch(console.error);
      }}

      document.getElementById("btn-download").addEventListener("click", function() {{
        const format = document.getElementById("format-select").value;
        downloadAs(format);
      }});

    }}).catch(console.error);
  </script>
</body>
</html>
"""

with open("categories_bar_chart.html", "w", encoding="utf-8") as f:
    f.write(html_template)

print("Done: categories_bar_chart.html")

Done: categories_bar_chart.html


In [20]:
with open("/content/drive/MyDrive/lmf/output/2026_08_31_web_visualizer/categories_bar_chart.html", "w", encoding="utf-8") as f:
    f.write(html_template)

# 5.0 Selection board - Accesion filter

In [26]:
# ============================================================
# Requirements
# ============================================================
# pip install pandas
#
# NOTE: uses Tabulator (https://tabulator.info) via CDN for the
# table — this is a data-grid task, not a chart.
#
# This assumes `df` already exists, with (at least) the columns:
# functional_group, lmf_category, quartile, id, id_lab, tax_name,
# methane_intensity, tddm, ch4_category, tddm_category,
# lmf_category_rank, quartile_rank.

import json
import pandas as pd

points_all = df.copy()

# ============================================================
# Column labels — human-readable titles shown in the table header
# (same mapping used in data_explorer.html, for consistency)
# ============================================================

column_labels = {
    "id": "Accession identifier",
    "id_lab": "Laboratory sample identifier",
    "tax_name": "Taxonomic name",
    "functional_group": "Functional group",
    "subset": "Experimental subset",
    "n_replicates_nutrition": "No. replicates (Nutrition)",
    "dm_percentage": "Dry matter (%)",
    "ash_dm": "Ash (% DM)",
    "om_percentage": "Organic matter (%)",
    "pc_percentage_dm": "Crude protein (% DM)",
    "adf_percentage_dm": "ADF (% DM)",
    "ndf_percentage_dm": "NDF (% DM)",
    "n_replicates_gas": "No. replicates (Gas)",
    "ch4_percentage_in_gas_8h": "Methane in Gas (8 h, %)",
    "ch4_percentage_in_gas_24h": "Methane in Gas (24 h, %)",
    "methane_intensity": "Methane intensity",
    "tddm": "TDDM (%)",
    "ch4_category": "Methane category",
    "tddm_category": "TDDM category",
    "lmf_category": "LMF category",
    "lmf_category_rank": "Position on LMF category",
    "quartile": "LMF quartile",
    "quartile_rank": "LMF ranking",
}


def display_title(col):
    """Human-readable title for a column: from column_labels if present
    (stripped of stray whitespace/tabs), else a title-cased fallback."""
    return column_labels.get(col, col.replace("_", " ").title()).strip()

# Ignore any functional_group values other than these three.
valid_functional_groups = ["Grasses", "Herbaceous_legumes", "Shrub_Trees"]
points_all = points_all[points_all["functional_group"].isin(valid_functional_groups)].copy()

# Columns shown in the results table/download, in this exact order.
# "id" (Accession identifier) is intentionally first.
DISPLAY_COLUMNS = [
    "id",
    "id_lab",
    "tax_name",
    "methane_intensity",
    "tddm",
    "ch4_category",
    "tddm_category",
    "lmf_category",
    "lmf_category_rank",
    "quartile",
    "quartile_rank",
]

# Columns used only for filtering (not displayed as their own
# table column, but kept in the row data so the filters can match
# against them).
FILTER_ONLY_COLUMNS = ["functional_group"]

missing = [c for c in DISPLAY_COLUMNS + FILTER_ONLY_COLUMNS if c not in points_all.columns]
if missing:
    raise ValueError(f"These expected columns are missing from df: {missing}")

# ============================================================
# Filter dropdown options
# ============================================================

functional_group_options = ["All"] + sorted(points_all["functional_group"].dropna().unique().tolist())
lmf_category_options = ["All"] + sorted(points_all["lmf_category"].dropna().unique().tolist())
quartile_options = ["All"] + sorted(points_all["quartile"].dropna().unique().tolist())

# ============================================================
# Data — keep display columns + the filter-only columns, NaN -> null
# ============================================================

data_df = points_all[DISPLAY_COLUMNS + FILTER_ONLY_COLUMNS].where(
    pd.notnull(points_all[DISPLAY_COLUMNS + FILTER_ONLY_COLUMNS]), None
)
records = data_df.to_dict(orient="records")
data_json = json.dumps(records)

# column_defs preserves DISPLAY_COLUMNS order, so "id" renders as
# the leftmost column in the table.
column_defs = [{"title": display_title(col), "field": col, "sorter": "string"} for col in DISPLAY_COLUMNS]
# Numeric columns get a number sorter instead.
NUMERIC_COLUMNS = {
    "methane_intensity", "tddm", "lmf_category_rank", "quartile_rank"
}
for c in column_defs:
    if c["field"] in NUMERIC_COLUMNS:
        c["sorter"] = "number"

columns_json = json.dumps(column_defs)


def options_json(options):
    return json.dumps(options)


html_template = f"""
<!DOCTYPE html>
<html>
<head>
  <meta charset="utf-8" />
  <title>Accession Filter</title>
  <link href="https://unpkg.com/tabulator-tables@5.5.2/dist/css/tabulator.min.css" rel="stylesheet">
  <script src="https://unpkg.com/tabulator-tables@5.5.2/dist/js/tabulator.min.js"></script>
  <style>
    * {{
      box-sizing: border-box;
    }}
    body {{
      font-family: 'Segoe UI', -apple-system, system-ui, Roboto, sans-serif;
      background: #f6f7fb;
      padding: 24px 16px;
      color: #2b2b33;
    }}
    .card {{
      background: white;
      border-radius: 14px;
      box-shadow: 0 2px 12px rgba(20, 20, 43, 0.06);
      padding: 24px 28px 28px;
      max-width: 1700px;
      margin: 0 auto;
    }}
    h2 {{
      margin: 0 0 4px;
      font-size: 22px;
      font-weight: 700;
      color: #1f1f2e;
    }}
    #meta {{
      color: #8a8a99;
      margin-bottom: 18px;
      font-size: 14px;
    }}
    #meta b {{
      color: #4a4a5a;
    }}

    #filters {{
      display: flex;
      flex-wrap: wrap;
      gap: 18px;
      margin-bottom: 18px;
      align-items: flex-end;
    }}
    .filter-group {{
      display: flex;
      flex-direction: column;
      gap: 4px;
    }}
    .filter-group label {{
      font-size: 12.5px;
      font-weight: 600;
      color: #6a6a78;
    }}
    .filter-group select {{
      padding: 8px 10px;
      border: 1px solid #e2e2ec;
      border-radius: 8px;
      font-size: 14px;
      min-width: 180px;
      background: white;
    }}

    #toolbar {{
      display: flex;
      gap: 10px;
      margin-bottom: 18px;
    }}
    #toolbar button {{
      padding: 9px 18px;
      font-size: 14px;
      font-weight: 600;
      border: none;
      border-radius: 8px;
      cursor: pointer;
    }}
    #btn-download-csv {{
      background: #4C6FFF;
      color: white;
      box-shadow: 0 2px 8px rgba(76, 111, 255, 0.35);
    }}
    #btn-download-csv:hover {{
      background: #3d5ce0;
    }}
    #btn-clear-filters {{
      background: #f1f1f5;
      color: #55555f;
    }}
    #btn-clear-filters:hover {{
      background: #e6e6ec;
    }}

    .tabulator {{
      border: none !important;
      background: transparent !important;
      font-size: 13.5px;
    }}
    .tabulator-header {{
      background: #fafafe !important;
      border-bottom: 2px solid #ececf3 !important;
    }}
    .tabulator-col {{
      background: #fafafe !important;
      border-right: none !important;
    }}
    .tabulator-col-title {{
      color: #3a3a45;
      font-weight: 600;
    }}
    .tabulator-row {{
      background: white !important;
      border-bottom: 1px solid #f0f0f5 !important;
    }}
    .tabulator-row:hover {{
      background: #f5f7ff !important;
    }}
    .tabulator-row.tabulator-row-even {{
      background: #fbfbfe !important;
    }}
    .tabulator-cell {{
      padding: 8px 10px !important;
    }}
    .tabulator-footer {{
      background: #fafafe !important;
      border-top: 2px solid #ececf3 !important;
    }}
    .tabulator-page.active {{
      background: #4C6FFF !important;
      color: white !important;
      border-color: #4C6FFF !important;
    }}
  </style>
</head>
<body>

  <div class="card">
    <h2>Accession Filter</h2>
    <div id="meta">Showing <b><span id="row-count">0</span></b> of <b>{len(records)}</b> accessions.</div>

    <div id="filters">
      <div class="filter-group">
        <label for="filter-group-select">Functional group</label>
        <select id="filter-group-select"></select>
      </div>
      <div class="filter-group">
        <label for="filter-category-select">LMF category</label>
        <select id="filter-category-select"></select>
      </div>
      <div class="filter-group">
        <label for="filter-quartile-select">Quartile</label>
        <select id="filter-quartile-select"></select>
      </div>
    </div>

    <div id="toolbar">
      <button id="btn-download-csv">⬇ Download filtered accessions (CSV)</button>
      <button id="btn-clear-filters">✕ Clear filters</button>
    </div>

    <div id="table"></div>
  </div>

  <script type="text/javascript">
    const tableData = {data_json};
    const columnDefs = {columns_json};

    const groupOptions = {options_json(functional_group_options)};
    const categoryOptions = {options_json(lmf_category_options)};
    const quartileOptions = {options_json(quartile_options)};

    function populateSelect(selectEl, options) {{
      options.forEach(function(opt) {{
        const o = document.createElement("option");
        o.value = opt;
        o.textContent = opt;
        selectEl.appendChild(o);
      }});
    }}

    const groupSelect = document.getElementById("filter-group-select");
    const categorySelect = document.getElementById("filter-category-select");
    const quartileSelect = document.getElementById("filter-quartile-select");

    populateSelect(groupSelect, groupOptions);
    populateSelect(categorySelect, categoryOptions);
    populateSelect(quartileSelect, quartileOptions);

    const table = new Tabulator("#table", {{
      data: tableData,
      columns: columnDefs,
      layout: "fitColumns",
      height: "600px",
      pagination: true,
      paginationSize: 25,
      paginationSizeSelector: [10, 25, 50, 100, true],
    }});

    function updateRowCount() {{
      document.getElementById("row-count").textContent = table.getDataCount("active");
    }}

    function applyFilters() {{
      const filters = [];
      if (groupSelect.value !== "All") {{
        filters.push({{ field: "functional_group", type: "=", value: groupSelect.value }});
      }}
      if (categorySelect.value !== "All") {{
        filters.push({{ field: "lmf_category", type: "=", value: categorySelect.value }});
      }}
      if (quartileSelect.value !== "All") {{
        filters.push({{ field: "quartile", type: "=", value: quartileSelect.value }});
      }}
      table.setFilter(filters);
    }}

    groupSelect.addEventListener("change", applyFilters);
    categorySelect.addEventListener("change", applyFilters);
    quartileSelect.addEventListener("change", applyFilters);

    table.on("tableBuilt", updateRowCount);
    table.on("dataFiltered", updateRowCount);

    // Build the CSV ourselves from the currently filtered ("active")
    // rows, using only the display columns (not the hidden
    // functional_group filter field).
    function toCsv(rows, columns) {{
      const fields = columns.map(c => c.field);
      const escape = (val) => {{
        if (val === null || val === undefined) return "";
        const s = String(val);
        if (/[",\\n]/.test(s)) {{
          return '"' + s.replace(/"/g, '""') + '"';
        }}
        return s;
      }};
      const header = fields.map(escape).join(",");
      const lines = rows.map(row => fields.map(f => escape(row[f])).join(","));
      return [header, ...lines].join("\\n");
    }}

    document.getElementById("btn-download-csv").addEventListener("click", function() {{
      const activeRows = table.getData("active");
      const csv = toCsv(activeRows, columnDefs);
      const blob = new Blob([csv], {{ type: "text/csv;charset=utf-8;" }});
      const url = URL.createObjectURL(blob);
      const link = document.createElement("a");
      link.href = url;
      link.download = "filtered_accessions.csv";
      document.body.appendChild(link);
      link.click();
      document.body.removeChild(link);
      URL.revokeObjectURL(url);
    }});

    document.getElementById("btn-clear-filters").addEventListener("click", function() {{
      groupSelect.value = "All";
      categorySelect.value = "All";
      quartileSelect.value = "All";
      table.setFilter([]);
    }});
  </script>
</body>
</html>
"""

with open("accession_filter.html", "w", encoding="utf-8") as f:
    f.write(html_template)

print("Done: accession_filter.html")

Done: accession_filter.html


In [27]:
with open("/content/drive/MyDrive/lmf/output/2026_08_31_web_visualizer/accession_filter.html", "w", encoding="utf-8") as f:
    f.write(html_template)

# 6.0 Filter and Save - Data Explorer

In [28]:
# ============================================================
# Requirements
# ============================================================
# pip install pandas
#
# NOTE: this uses Tabulator (https://tabulator.info), a JS table
# library loaded via CDN — no Vega-Lite involved here, since this
# is a data-grid task, not a chart.
#
# This assumes `df` already exists (the same dataframe you've been
# using for the pie/banner/bar chart).

import json
import math
import pandas as pd

points_all = df.copy()

# Put "id" (Accession identifier) first, keeping every other column
# in its original relative order after it.
_cols = list(points_all.columns)
if "id" in _cols:
    _cols.remove("id")
    _cols = ["id"] + _cols
    points_all = points_all[_cols]

# ============================================================
# Column labels — human-readable titles shown in the table header
# ============================================================

column_labels = {
    "id": "Accession identifier",
    "id_lab": "Laboratory sample identifier",
    "tax_name": "Taxonomic name",
    "functional_group": "Functional group",
    "subset": "Experimental subset",
    "n_replicates_nutrition": "No. replicates (Nutrition)",
    "dm_percentage": "Dry matter (%)",
    "ash_dm": "Ash (% DM)",
    "om_percentage": "Organic matter (%)",
    "pc_percentage_dm": "Crude protein (% DM)",
    "adf_percentage_dm": "ADF (% DM)",
    "ndf_percentage_dm": "NDF (% DM)",
    "n_replicates_gas": "No. replicates (Gas)",
    "ch4_percentage_in_gas_8h": "Methane in Gas (8 h, %)",
    "ch4_percentage_in_gas_24h": "Methane in Gas (24 h, %)",
    "methane_intensity": "Methane intensity",
    "tddm": "TDDM (%)",
    "ch4_category": "Methane category",
    "tddm_category": "TDDM category",
    "lmf_category": "LMF category",
    "lmf_category_rank": "Position on LMF category",
    "quartile": "LMF quartile",
    "quartile_rank": "LMF ranking",
}


def display_title(col):
    """Human-readable title for a column: from column_labels if present
    (stripped of stray whitespace/tabs), else a title-cased fallback."""
    return column_labels.get(col, col.replace("_", " ").title()).strip()

# ============================================================
# Decide, per column, what kind of filter makes sense:
# - Numeric columns  -> a "≥" number filter (type a number, see
#   rows with that value or higher)
# - Low-cardinality text/category columns (<= 20 unique values)
#   -> a dropdown ("select") filter
# - Everything else (e.g. free-text columns like tax_name/id)
#   -> a "contains" text filter
# ============================================================

MAX_CATEGORICAL_UNIQUE = 20

column_defs = []
for col in points_all.columns:
    series = points_all[col]
    is_numeric = pd.api.types.is_numeric_dtype(series)

    if is_numeric:
        column_defs.append({
            "title": display_title(col),
            "field": col,
            "headerFilter": "input",
            "headerFilterFunc": ">=",
            "headerFilterPlaceholder": "≥ value",
            "sorter": "number"
        })
    else:
        n_unique = series.dropna().nunique()
        if n_unique <= MAX_CATEGORICAL_UNIQUE:
            options = ["(All)"] + sorted(series.dropna().unique().tolist())
            column_defs.append({
                "title": display_title(col),
                "field": col,
                "headerFilter": "list",
                "headerFilterParams": {"values": options, "clearable": True},
                "headerFilterFunc": "=",
                "sorter": "string"
            })
        else:
            column_defs.append({
                "title": display_title(col),
                "field": col,
                "headerFilter": "input",
                "headerFilterFunc": "like",
                "headerFilterPlaceholder": "contains...",
                "sorter": "string"
            })

# ============================================================
# Data — NaN -> null, so it serializes cleanly to JSON
# ============================================================

records = points_all.where(pd.notnull(points_all), None).to_dict(orient="records")

data_json = json.dumps(records)
columns_json = json.dumps(column_defs)

# ============================================================
# Header accent colors for specific column groups
# ============================================================

COLUMN_COLORS = {
    "id": ("#EEF1FF", "#3B4FCC"),           # indigo — identifier
    "dm_percentage": ("#EAF7EE", "#2F855A"),      # green — nutrition composition
    "ash_dm": ("#EAF7EE", "#2F855A"),
    "om_percentage": ("#EAF7EE", "#2F855A"),
    "pc_percentage_dm": ("#EAF7EE", "#2F855A"),
    "adf_percentage_dm": ("#EAF7EE", "#2F855A"),
    "ndf_percentage_dm": ("#EAF7EE", "#2F855A"),
    "methane_intensity": ("#FFF3E6", "#B15C00"),  # orange — methane metrics
    "tddm": ("#FFF3E6", "#B15C00"),
    "lmf_category": ("#F3EAFB", "#7B2FBE"),       # purple — classification
}

column_color_css = "\n".join(
    f"""
    .tabulator-col[tabulator-field="{field}"] {{
      background: {bg} !important;
    }}
    .tabulator-col[tabulator-field="{field}"] .tabulator-col-title {{
      color: {fg} !important;
      font-weight: 700 !important;
    }}"""
    for field, (bg, fg) in COLUMN_COLORS.items()
)

# ============================================================
# Build the HTML
# ============================================================

html_template = f"""
<!DOCTYPE html>
<html>
<head>
  <meta charset="utf-8" />
  <title>Data Explorer</title>
  <link href="https://unpkg.com/tabulator-tables@5.5.2/dist/css/tabulator.min.css" rel="stylesheet">
  <script src="https://unpkg.com/tabulator-tables@5.5.2/dist/js/tabulator.min.js"></script>
  <style>
    * {{
      box-sizing: border-box;
    }}
    body {{
      font-family: 'Segoe UI', -apple-system, system-ui, Roboto, sans-serif;
      background: #f6f7fb;
      padding: 32px;
      color: #2b2b33;
    }}
    .card {{
      background: white;
      border-radius: 14px;
      box-shadow: 0 2px 12px rgba(20, 20, 43, 0.06);
      padding: 24px 28px 28px;
      max-width: 1400px;
      margin: 0 auto;
    }}
    h2 {{
      margin: 0 0 4px;
      font-size: 22px;
      font-weight: 700;
      color: #1f1f2e;
    }}
    #meta {{
      color: #8a8a99;
      margin-bottom: 18px;
      font-size: 14px;
    }}
    #meta b {{
      color: #4a4a5a;
    }}
    #toolbar {{
      display: flex;
      gap: 10px;
      margin-bottom: 18px;
    }}
    #toolbar button {{
      padding: 9px 18px;
      font-size: 14px;
      font-weight: 600;
      border: none;
      border-radius: 8px;
      cursor: pointer;
      transition: transform 0.05s ease, box-shadow 0.15s ease;
    }}
    #btn-download-csv {{
      background: #4C6FFF;
      color: white;
      box-shadow: 0 2px 8px rgba(76, 111, 255, 0.35);
    }}
    #btn-download-csv:hover {{
      background: #3d5ce0;
    }}
    #btn-clear-filters {{
      background: #f1f1f5;
      color: #55555f;
    }}
    #btn-clear-filters:hover {{
      background: #e6e6ec;
    }}
    #toolbar button:active {{
      transform: scale(0.98);
    }}

    /* ---- Tabulator theme overrides: cleaner, less gray ---- */
    .tabulator {{
      border: none !important;
      background: transparent !important;
      font-size: 13.5px;
    }}
    .tabulator-header {{
      background: #fafafe !important;
      border-bottom: 2px solid #ececf3 !important;
    }}
    .tabulator-col {{
      background: #fafafe !important;
      border-right: none !important;
    }}
    .tabulator-col-title {{
      color: #3a3a45;
      font-weight: 600;
    }}
    .tabulator-row {{
      background: white !important;
      border-bottom: 1px solid #f0f0f5 !important;
    }}
    .tabulator-row:hover {{
      background: #f5f7ff !important;
    }}
    .tabulator-row.tabulator-row-even {{
      background: #fbfbfe !important;
    }}
    .tabulator-cell {{
      padding: 8px 10px !important;
    }}
    .tabulator-header-filter input,
    .tabulator-header-filter select {{
      border: 1px solid #e2e2ec !important;
      border-radius: 6px !important;
      padding: 4px 6px !important;
    }}
    .tabulator-footer {{
      background: #fafafe !important;
      border-top: 2px solid #ececf3 !important;
    }}
    .tabulator-paginator {{
      color: #55555f;
    }}
    .tabulator-page {{
      border-radius: 6px !important;
    }}
    .tabulator-page.active {{
      background: #4C6FFF !important;
      color: white !important;
      border-color: #4C6FFF !important;
    }}

    /* ---- Per-column header accent colors ---- */
    {column_color_css}
  </style>
</head>
<body>

  <div class="card">
    <h2>Data Explorer</h2>
    <div id="meta">Showing <b><span id="row-count">0</span></b> of <b>{len(records)}</b> rows — use the boxes under each column header to filter.</div>

    <div id="toolbar">
      <button id="btn-download-csv">⬇ Download filtered data (CSV)</button>
      <button id="btn-clear-filters">✕ Clear all filters</button>
    </div>

    <div id="table"></div>
  </div>

  <script type="text/javascript">
    const tableData = {data_json};
    const columnDefs = {columns_json};

    const table = new Tabulator("#table", {{
      data: tableData,
      columns: columnDefs,
      layout: "fitDataFill",
      height: "650px",
      pagination: true,
      paginationSize: 25,
      paginationSizeSelector: [10, 25, 50, 100, true],
    }});

    function updateRowCount() {{
      document.getElementById("row-count").textContent = table.getDataCount("active");
    }}

    table.on("tableBuilt", updateRowCount);
    table.on("dataFiltered", updateRowCount);

    // Build the CSV ourselves from the currently filtered ("active")
    // rows, so the download always matches exactly what's on screen.
    function toCsv(rows, columns) {{
      const fields = columns.map(c => c.field);
      const escape = (val) => {{
        if (val === null || val === undefined) return "";
        const s = String(val);
        if (/[",\\n]/.test(s)) {{
          return '"' + s.replace(/"/g, '""') + '"';
        }}
        return s;
      }};
      const header = fields.map(escape).join(",");
      const lines = rows.map(row => fields.map(f => escape(row[f])).join(","));
      return [header, ...lines].join("\\n");
    }}

    document.getElementById("btn-download-csv").addEventListener("click", function() {{
      const activeRows = table.getData("active");
      const csv = toCsv(activeRows, columnDefs);
      const blob = new Blob([csv], {{ type: "text/csv;charset=utf-8;" }});
      const url = URL.createObjectURL(blob);
      const link = document.createElement("a");
      link.href = url;
      link.download = "filtered_data.csv";
      document.body.appendChild(link);
      link.click();
      document.body.removeChild(link);
      URL.revokeObjectURL(url);
    }});

    document.getElementById("btn-clear-filters").addEventListener("click", function() {{
      table.clearHeaderFilter();
    }});
  </script>
</body>
</html>
"""

with open("data_explorer.html", "w", encoding="utf-8") as f:
    f.write(html_template)

print("Done: data_explorer.html")

Done: data_explorer.html


In [24]:
with open("/content/drive/MyDrive/lmf/output/2026_08_31_web_visualizer/data_explorer.html", "w", encoding="utf-8") as f:
    f.write(html_template)

# 7.0 Data Visualization

In [30]:
# ---------------------------------------------
# Lists of special samples
# ---------------------------------------------

target = ["CIAT-PM-21-2580"]

star_grass_control_ids = ["Sample-8"]

benchmark_ids = ["CIAT-6962-Mombaza-Ex","CIAT-606-Basilisk-Opt","CIAT-6294-Marandu-Opt","CIAT-36087-MulatoII-Opt",
                 "CIAT-6133-Llanero-Opt","CIAT-6962-Mombaza-Opt","Paja-Sabana-Madura","Humidicola-679","Sabana-Quemada",
                 "CIAT-6294-Marandu-Exc", "CIAT-606-Basilisk-Exc", "BR-02-1752-Cayman-Exc", "BR-06-423-Cayman-Exc"]

## 7.1 Scatterplot

In [32]:
# ============================================================
# Requirements
# ============================================================
# pip install altair vl-convert-python pandas numpy
#
# vl-convert-python is required so that chart.to_json()-based
# rendering works cleanly.

import json
import numpy as np
import pandas as pd
import altair as alt

# Disable Altair's default 5000-row safety limit, in case df is large.
alt.data_transformers.disable_max_rows()

# ============================================================
# NOTE: this assumes `df` (with "id", "id_lab", "subset",
# "functional_group", "lmf_category", "quartile", and various
# numeric columns) already exists, exactly like in the original
# script. It also assumes `benchmark_ids`, `star_grass_control_ids`,
# and `target` (lists of id values) already exist.
# ============================================================

# ============================================================
# Column labels
# ============================================================

column_labels = {
    "id": "Accession identifier",
    "id_lab": "Laboratory sample identifier",
    "tax_name": "Taxonomic name",
    "functional_group": "Functional group",
    "subset": "Experimental subset",
    "n_replicates_nutrition": "No. replicates (Nutrition)",
    "dm_percentage": "Dry matter (%)",
    "ash_dm": "Ash (% DM)",
    "om_percentage": "Organic matter (%)",
    "pc_percentage_dm": "Crude protein (% DM)",
    "adf_percentage_dm": "ADF (% DM)",
    "ndf_percentage_dm": "NDF (% DM)",
    "n_replicates_gas": "No. replicates (Gas)",
    "ch4_percentage_in_gas_8h": "Methane in Gas (8 h, %)",
    "ch4_percentage_in_gas_24h": "Methane in Gas (24 h, %)",
    "methane_intensity": "Methane intensity",
    "tddm": "TDDM (%)",
    "ch4_category": "Methane category",
    "tddm_category": "TDDM category",
    "lmf_category": "LMF category",
    "lmf_category_rank": "Position on LMF category",
    "quartile": "LMF quartile",
    "quartile_rank": "LMF ranking",
}

# ============================================================
# Numeric columns available for the X / Y dropdowns
# ============================================================

numeric_columns = sorted(df.select_dtypes(include="number").columns.tolist())

default_x = "tddm"
default_y = "methane_intensity"

if default_x not in numeric_columns:
    default_x = numeric_columns[0]

if default_y not in numeric_columns:
    default_y = numeric_columns[1] if len(numeric_columns) > 1 else numeric_columns[0]

# Pretty labels for each numeric column, from column_labels when
# available (falling back to a title-cased version of the column
# name) — used both for the dropdown option labels and for the
# best-effort dynamic axis title update in the browser.
pretty_labels = {
    col: column_labels.get(col, col.replace("_", " ").title())
    for col in numeric_columns
}

# ============================================================
# Tooltip columns
# ============================================================

tooltip_columns = [
    "id", "id_lab", "tax_name", "functional_group", "subset",
    "n_replicates_nutrition", "dm_percentage", "ash_dm", "om_percentage",
    "pc_percentage_dm", "adf_percentage_dm", "ndf_percentage_dm",
    "n_replicates_gas", "ch4_percentage_in_gas_8h", "ch4_percentage_in_gas_24h",
    "methane_intensity", "tddm", "ch4_category", "tddm_category",
    "lmf_category", "lmf_category_rank", "quartile", "quartile_rank"
]

# ============================================================
# Plot group (color category) — computed once for the whole
# dataset, since it doesn't depend on the subset/group/category/
# quartile filters. Note: membership is checked against "id" here
# (not "id_lab"), matching the updated source script.
# ============================================================

points_all = df.copy()
points_all["plot_group"] = points_all["functional_group"]

points_all.loc[points_all["id"].isin(benchmark_ids), "plot_group"] = "Benchmark"
points_all.loc[points_all["id"].isin(star_grass_control_ids), "plot_group"] = "Star Grass Control"
points_all.loc[points_all["id"].isin(target), "plot_group"] = "Target"

plot_group_order = [
    "Herbaceous_legumes",
    "Grasses",
    "Shrub_Trees",
    "Benchmark",
    "Star Grass Control",
]

plot_group_colors = [
    "green",
    "cornflowerblue",
    "orange",
    "purple",
    "black",
]

# ============================================================
# Dropdown options for subset / functional group / category / quartile
# ============================================================

subset_options = ["All"] + sorted(points_all["subset"].astype(str).unique().tolist())
valid_functional_groups = ["Grasses", "Herbaceous_legumes", "Shrub_Trees"]
functional_group_options = ["All"] + valid_functional_groups
category_options = ["All"] + sorted(points_all["lmf_category"].dropna().unique().tolist())
quartile_options = ["All"] + sorted(points_all["quartile"].dropna().unique().tolist())

# ============================================================
# Interactive controls
# ============================================================
# X/Y dropdowns show pretty labels but carry the actual column name
# as the underlying value, using Altair's separate options/labels
# arguments for binding_select.

x_options = [(pretty_labels[col], col) for col in numeric_columns]
y_options = [(pretty_labels[col], col) for col in numeric_columns]

sel_x = alt.param(
    name="sel_x",
    value=default_x,
    bind=alt.binding_select(options=[c for _, c in x_options], labels=[l for l, _ in x_options], name="X: ")
)

sel_y = alt.param(
    name="sel_y",
    value=default_y,
    bind=alt.binding_select(options=[c for _, c in y_options], labels=[l for l, _ in y_options], name="Y: ")
)

sel_subset = alt.param(
    name="sel_subset",
    value="All",
    bind=alt.binding_select(options=subset_options, name="Subset: ")
)

sel_group = alt.param(
    name="sel_group",
    value="All",
    bind=alt.binding_select(options=functional_group_options, name="Functional group: ")
)

sel_category = alt.param(
    name="sel_category",
    value="All",
    bind=alt.binding_select(options=category_options, name="Category: ")
)

sel_quartile = alt.param(
    name="sel_quartile",
    value="All",
    bind=alt.binding_select(options=quartile_options, name="Quartile: ")
)

# ------------------------------------------------------------
# Text search box: highlight by id or id_lab, comma-separated,
# partial (substring) match, case-insensitive
# ------------------------------------------------------------

search_param = alt.param(
    name="search_id",
    value="",
    bind=alt.binding(input="text", name="Search ID or Lab ID (comma-separated): ")
)

search_pattern_expr = (
    "replace(replace(search_id, /\\s+/g, ''), /,/g, '|')"
)

is_match_expr = (
    f"search_id !== '' && ("
    f"test(regexp({search_pattern_expr}, 'i'), toString(datum.id)) || "
    f"test(regexp({search_pattern_expr}, 'i'), toString(datum.id_lab))"
    f")"
)

filter_expr = (
    "(sel_subset == 'All' || toString(datum.subset) == sel_subset) && "
    "(sel_group == 'All' || datum.functional_group == sel_group) && "
    "(sel_category == 'All' || datum.lmf_category == sel_category) && "
    "(sel_quartile == 'All' || datum.quartile == sel_quartile)"
)

tooltip = [
    alt.Tooltip(col, title=column_labels.get(col, col.replace("_", " ").title()))
    for col in tooltip_columns
    if col in points_all.columns
] + [
    alt.Tooltip("x_value:Q", title="X"),
    alt.Tooltip("y_value:Q", title="Y"),
]

# ------------------------------------------------------------
# Title row (guaranteed fallback — always shows a readable label
# for the selected columns, regardless of whether the JS-based
# axis title update below works in a given browser)
# ------------------------------------------------------------

pretty_labels_json = json.dumps(pretty_labels)

title_chart = (
    alt.Chart(pd.DataFrame({"_dummy": [1]}))
    .transform_calculate(
        label=f"({pretty_labels_json})[sel_y] + ' vs ' + ({pretty_labels_json})[sel_x]"
    )
    .mark_text(align="center", fontSize=18, fontWeight="bold", dy=0)
    .encode(text="label:N")
    .properties(width=850, height=30)
)

# ------------------------------------------------------------
# Scatter points — base layer (all non-matched points, by
# plot_group) and highlight layer (only matched points, one
# shape per id)
# ------------------------------------------------------------

base_layer = (
    alt.Chart(points_all)
    .mark_circle(size=70)
    .transform_calculate(
        x_value="datum[sel_x]",
        y_value="datum[sel_y]",
        # 'Target' is no longer a category in the color scale — these
        # rows just take their functional_group's normal color instead.
        display_group="datum.plot_group === 'Target' ? datum.functional_group : datum.plot_group"
    )
    .transform_filter(filter_expr)
    .transform_filter(f"!({is_match_expr})")
    .encode(
        x=alt.X("x_value:Q", title=pretty_labels[default_x]),
        y=alt.Y("y_value:Q", title=pretty_labels[default_y]),
        color=alt.Color(
            "display_group:N",
            scale=alt.Scale(domain=plot_group_order, range=plot_group_colors),
            legend=alt.Legend(title="Sample Type", symbolSize=70)
        ),
        tooltip=tooltip
    )
)

highlight_layer = (
    alt.Chart(points_all)
    .mark_point(filled=True, size=70, stroke="black", strokeWidth=1.5)
    .transform_calculate(
        x_value="datum[sel_x]",
        y_value="datum[sel_y]"
    )
    .transform_filter(filter_expr)
    .transform_filter(is_match_expr)
    .encode(
        x=alt.X("x_value:Q"),
        y=alt.Y("y_value:Q"),
        # Only matched rows are ever present in this layer's data, so the
        # shape scale's domain — and therefore the legend — automatically
        # lists exactly the searched accession(s), and only those.
        shape=alt.Shape(
            "id:N",
            legend=alt.Legend(title="Highlighted accession(s)", symbolSize=70, titleLimit=0)
        ),
        color=alt.value("red"),
        tooltip=tooltip
    )
)

points_chart = base_layer + highlight_layer

biplot = points_chart.properties(width=850, height=600)

# ------------------------------------------------------------
# Final layout — controls above, then title, then the plot
# ------------------------------------------------------------

final_plot = (
    alt.vconcat(title_chart, biplot)
    .add_params(sel_x, sel_y, sel_subset, sel_group, sel_category, sel_quartile, search_param)
    .configure_axis(labelFontSize=14, titleFontSize=18)
    .configure_legend(titleFontSize=16, labelFontSize=14)
    .configure_view(strokeWidth=0)
    .interactive()
)

# ============================================================
# Save as interactive HTML with controls forced above the chart
# ============================================================

chart_spec = final_plot.to_json(indent=None)
pretty_labels_json = json.dumps(pretty_labels)

html_template = f"""
<!DOCTYPE html>
<html>
<head>
  <meta charset="utf-8" />
  <title>Interactive Scatterplot</title>
  <script src="https://cdn.jsdelivr.net/npm/vega@5"></script>
  <script src="https://cdn.jsdelivr.net/npm/vega-lite@5"></script>
  <script src="https://cdn.jsdelivr.net/npm/vega-embed@6"></script>
  <script src="https://cdn.jsdelivr.net/npm/utif@3.1.0/UTIF.js"></script>
  <script src="https://cdnjs.cloudflare.com/ajax/libs/jspdf/2.5.1/jspdf.umd.min.js"></script>
  <style>
    body {{
      font-family: sans-serif;
      display: flex;
      flex-direction: column;
      align-items: center;
    }}
    #controls {{
      margin: 20px 0;
      font-size: 16px;
    }}
    /* Visual gap between the X/Y axis controls and the rest
       (Subset, Group, Category, Quartile, Search). Controls are
       added in that order, so the 3rd one (Subset) gets pushed
       right to create the separation. Targeting .vega-bind directly
       since bindingsElement points straight at #controls (no
       intermediate .vega-bindings wrapper in that case). */
    #controls .vega-bind:nth-of-type(3) {{
      margin-left: 28px;
    }}
    #controls .vega-bindings .vega-bind:nth-of-type(3) {{
      margin-left: 28px;
    }}
    #controls > div:nth-of-type(3) {{
      margin-left: 28px;
    }}
    #download-buttons {{
      margin: 20px 0 10px 0;
      display: flex;
      align-items: center;
      gap: 10px;
    }}
    #download-buttons select,
    #download-buttons button {{
      padding: 8px 16px;
      font-size: 14px;
      border: 1px solid #ccc;
      border-radius: 6px;
      background: #f7f7f9;
      cursor: pointer;
    }}
    #download-buttons button:hover {{
      background: #e9e9ee;
    }}
  </style>
</head>
<body>
  <div id="controls"></div>
  <div id="vis"></div>
  <div id="download-buttons">
    <select id="format-select">
      <option value="png">PNG</option>
      <option value="jpg">JPG</option>
      <option value="tiff">TIFF</option>
      <option value="pdf">PDF</option>
    </select>
    <button id="btn-download">Download</button>
  </div>

  <script type="text/javascript">
    const spec = {chart_spec};
    const prettyLabels = {pretty_labels_json};

    const DPI = 300;
    const DPI_SCALE = DPI / 96;

    vegaEmbed("#vis", spec, {{
      actions: false,
      bindingsElement: "#controls",
      renderer: "svg"
    }}).then(function(result) {{

      const view = result.view;

      function triggerDownload(url, filename) {{
        const link = document.createElement("a");
        link.href = url;
        link.download = filename;
        document.body.appendChild(link);
        link.click();
        document.body.removeChild(link);
      }}

      function downloadAs(format) {{
        view.toCanvas(DPI_SCALE).then(function(canvas) {{

          if (format === "png") {{
            triggerDownload(canvas.toDataURL("image/png"), "scatterplot.png");

          }} else if (format === "jpg") {{
            triggerDownload(canvas.toDataURL("image/jpeg", 0.95), "scatterplot.jpg");

          }} else if (format === "tiff") {{
            const ctx = canvas.getContext("2d");
            const imageData = ctx.getImageData(0, 0, canvas.width, canvas.height);
            const tiffBuffer = UTIF.encodeImage(imageData.data, canvas.width, canvas.height);
            const blob = new Blob([tiffBuffer], {{ type: "image/tiff" }});
            const url = URL.createObjectURL(blob);
            triggerDownload(url, "scatterplot.tiff");

          }} else if (format === "pdf") {{
            const widthIn = canvas.width / DPI;
            const heightIn = canvas.height / DPI;
            const {{ jsPDF }} = window.jspdf;
            const pdf = new jsPDF({{
              orientation: widthIn >= heightIn ? "landscape" : "portrait",
              unit: "in",
              format: [widthIn, heightIn]
            }});
            pdf.addImage(canvas.toDataURL("image/png"), "PNG", 0, 0, widthIn, heightIn);
            pdf.save("scatterplot.pdf");
          }}

        }}).catch(console.error);
      }}

      document.getElementById("btn-download").addEventListener("click", function() {{
        const format = document.getElementById("format-select").value;
        downloadAs(format);
      }});

      // Explicitly hide the "Highlighted accession(s)" legend while the
      // search box is empty.
      function toggleHighlightedAccessionsLegend(searchValue) {{
        const container = view.container();
        if (!container) return;

        const isActive = !!(searchValue && searchValue.trim() !== "");

        const legendGroups = container.querySelectorAll('[aria-roledescription="legend"]');
        legendGroups.forEach(function(g) {{
          const texts = g.querySelectorAll("text");
          let isTargetLegend = false;
          texts.forEach(function(t) {{
            if (t.textContent.trim().indexOf("Highlighted") === 0) {{
              isTargetLegend = true;
            }}
          }});
          if (isTargetLegend) {{
            g.style.display = isActive ? "" : "none";
          }}
        }});
      }}

      // BEST EFFORT: update the actual x/y axis titles to show the
      // pretty column label (from column_labels) for whatever is
      // currently selected. The raw "Y vs X" text is ALSO shown in
      // the chart's main title above, which does not depend on this
      // and always works.
      function updateAxisTitles() {{
        const xCol = view.signal("sel_x");
        const yCol = view.signal("sel_y");
        const xLabel = prettyLabels[xCol] || xCol;
        const yLabel = prettyLabels[yCol] || yCol;

        const container = view.container();
        if (!container) return;

        // Walk up from a text node toward (but not including) `root`,
        // returning true if any ancestor along the way looks like a
        // tick/label group (as opposed to the axis title itself).
        function isInsideTickLabelGroup(el, root) {{
          let node = el.parentElement;
          while (node && node !== root) {{
            const cls = node.getAttribute("class") || "";
            if (cls.indexOf("label") !== -1 || cls.indexOf("tick") !== -1) {{
              return true;
            }}
            node = node.parentElement;
          }}
          return false;
        }}

        const titleEls = new Set();

        // Priority path: elements explicitly classed as the title.
        container.querySelectorAll("text.role-axis-title").forEach(function(el) {{
          titleEls.add(el);
        }});

        // General path: any text inside an axis group that is NOT
        // inside a tick/label sub-group, regardless of nesting depth.
        const axisRoots = container.querySelectorAll('[aria-roledescription="axis"], g.role-axis');
        axisRoots.forEach(function(root) {{
          root.querySelectorAll("text").forEach(function(t) {{
            if (!isInsideTickLabelGroup(t, root)) {{
              titleEls.add(t);
            }}
          }});
        }});

        titleEls.forEach(function(t) {{
          const transform = t.getAttribute("transform") || "";
          const isRotated = transform.indexOf("rotate") !== -1;
          t.textContent = isRotated ? yLabel : xLabel;
        }});
      }}

      toggleHighlightedAccessionsLegend(view.signal("search_id"));
      view.addSignalListener("search_id", function(name, value) {{
        setTimeout(function() {{ toggleHighlightedAccessionsLegend(value); }}, 50);
      }});

      setTimeout(updateAxisTitles, 100);
      view.addSignalListener("sel_x", function() {{
        setTimeout(updateAxisTitles, 100);
        setTimeout(updateAxisTitles, 300);
      }});
      view.addSignalListener("sel_y", function() {{
        setTimeout(updateAxisTitles, 100);
        setTimeout(updateAxisTitles, 300);
      }});

    }}).catch(console.error);
  </script>
</body>
</html>
"""

with open("scatterplot.html", "w", encoding="utf-8") as f:
    f.write(html_template)

print("Done: scatterplot.html")

Done: scatterplot.html


In [33]:
with open("/content/drive/MyDrive/lmf/output/2026_08_31_web_visualizer/scatterplot.html", "w", encoding="utf-8") as f:
    f.write(html_template)

## 7.2 PCA

In [37]:
import ipywidgets as widgets

In [38]:
# ============================================================
# PCA variables
# ============================================================

pca_vars = [
    "dm_percentage",
    "ash_dm",
    "om_percentage",
    "pc_percentage_dm",
    "adf_percentage_dm",
    "ndf_percentage_dm",
    "ch4_percentage_in_gas_8h",
    "ch4_percentage_in_gas_24h",
    "methane_intensity",
    "tddm"
]


# ============================================================
# Column labels
# ============================================================

column_labels = {
    "id": "Accession identifier",
    "id_lab": "Laboratory sample identifier",
    "tax_name": "Taxonomic name",
    "functional_group": "Functional group",
    "point_category": "Plot category",
    "subset": "Experimental subset",
    "n_replicates_nutrition": "No. replicates (Nutrition)",
    "dm_percentage": "Dry matter (%)",
    "ash_dm": "Ash (% DM)",
    "om_percentage": "Organic matter (% DM)",
    "pc_percentage_dm": "Crude protein (% DM)",
    "adf_percentage_dm": "ADF (% DM)",
    "ndf_percentage_dm": "NDF (% DM)",
    "n_replicates_gas": "No. replicates (Gas)",
    "ch4_percentage_in_gas_8h": "Methane in Gas (8 h, %)",
    "ch4_percentage_in_gas_24h": "Methane in Gas (24 h, %)",
    "methane_intensity": "Methane intensity",
    "tddm": "TDDM (%)",
    "ch4_category": "Methane category",
    "tddm_category": "TDDM category",
    "lmf_category": "LMF category",
    "lmf_category_rank": "Position on LMF category",
    "quartile": "LMF quartile",
    "quartile_rank": "LMF ranking"
}


# ============================================================
# Tooltip columns
# ============================================================

tooltip_columns = [
    "id",
    "id_lab",
    "tax_name",
    "functional_group",
    "point_category",
    "subset",
    "n_replicates_nutrition",
    "dm_percentage",
    "ash_dm",
    "om_percentage",
    "pc_percentage_dm",
    "adf_percentage_dm",
    "ndf_percentage_dm",
    "n_replicates_gas",
    "ch4_percentage_in_gas_8h",
    "ch4_percentage_in_gas_24h",
    "methane_intensity",
    "tddm",
    "ch4_category",
    "tddm_category",
    "lmf_category",
    "lmf_category_rank",
    "quartile",
    "quartile_rank"
]


# ============================================================
# Check PCA variables
# ============================================================

missing_pca_vars = [
    column
    for column in pca_vars
    if column not in df.columns
]

if missing_pca_vars:

    raise ValueError(
        "The following PCA variables are missing from df: "
        f"{missing_pca_vars}"
    )


# ============================================================
# Clean functional group
# ============================================================

df["functional_group"] = (
    df["functional_group"]
    .astype("string")
    .str.strip()
)


# ============================================================
# Valid functional groups
# ============================================================

valid_functional_groups = [
    "Grasses",
    "Herbaceous_legumes",
    "Shrub_Trees"
]


# ============================================================
# Filter data for PCA
# ============================================================

df_pca = df[
    df["functional_group"].isin(
        valid_functional_groups
    )
].copy()


# ============================================================
# Special sample categories
# ============================================================

# Default category = functional group
df_pca["point_category"] = (
    df_pca["functional_group"]
)


# ------------------------------------------------------------
# Benchmark samples
# ------------------------------------------------------------

df_pca.loc[
    df_pca["id"].isin(benchmark_ids),
    "point_category"
] = "Benchmark"


# ------------------------------------------------------------
# Star grass control
# ------------------------------------------------------------

df_pca.loc[
    df_pca["id"].isin(star_grass_control_ids),
    "point_category"
] = "Star grass control"


# ------------------------------------------------------------
# Target sample
# ------------------------------------------------------------

#df_pca.loc[df_pca["id"].isin(target),"point_category"] = "Target"


# ============================================================
# Functional groups available for widget
# ============================================================

functional_groups = sorted(
    df_pca["functional_group"]
    .dropna()
    .unique()
    .tolist()
)


# ============================================================
# Widget
# ============================================================

pca_functional_widget = widgets.Dropdown(
    options=["All"] + functional_groups,
    value="All",
    description="Group:"
)


# ============================================================
# Point category order
# ============================================================

point_category_order = [
    "Grasses",
    "Herbaceous_legumes",
    "Shrub_Trees",
    "Benchmark",
    "Star grass control",
    "Target"
]


# ============================================================
# Point category colors
# ============================================================

point_category_colors = [
    "#4C78A8",   # Grasses
    "#59A14F",   # Herbaceous legumes
    "#F28E2B",   # Shrub/Trees
    "purple",   # Benchmark
    "black",   # Star grass control
    "red"    # Target
]


# ============================================================
# PCA function
# ============================================================

def plot_pca(functional_group):


    # ========================================================
    # Filter data by functional group
    # ========================================================

    if functional_group == "All":

        data = df_pca.copy()

    else:

        data = df_pca[
            df_pca["functional_group"]
            == functional_group
        ].copy()


    # ========================================================
    # Check number of observations
    # ========================================================

    if len(data) < 3:

        print(
            f"Not enough observations for "
            f"'{functional_group}'. "
            f"Found {len(data)} samples."
        )

        return


    # ========================================================
    # PCA matrix
    # ========================================================

    X = data[pca_vars].copy()


    # ========================================================
    # Replace infinite values with NaN
    # ========================================================

    X.replace(
        [np.inf, -np.inf],
        np.nan,
        inplace=True
    )


    # ========================================================
    # Impute missing values with column means
    # ========================================================

    X = X.fillna(
        X.mean()
    )


    # ========================================================
    # Check remaining NaNs
    # ========================================================

    if X.isna().any().any():

        bad_columns = X.columns[
            X.isna().any()
        ].tolist()

        print(
            "These variables still contain missing values: "
            f"{bad_columns}"
        )

        return


    # ========================================================
    # Check zero variance
    # ========================================================

    zero_variance = [
        column
        for column in pca_vars
        if X[column].nunique() <= 1
    ]

    if zero_variance:

        print(
            "These variables have zero variance for "
            f"'{functional_group}': "
            f"{zero_variance}"
        )

        return


    # ========================================================
    # Standardization
    # ========================================================

    scaler = StandardScaler()

    X_scaled = scaler.fit_transform(
        X
    )


    # ========================================================
    # PCA
    # ========================================================

    pca = PCA(
        n_components=2
    )

    principal_components = pca.fit_transform(
        X_scaled
    )


    # ========================================================
    # PCA scores
    # ========================================================

    pca_df = pd.DataFrame(
        principal_components,
        columns=[
            "PC1",
            "PC2"
        ],
        index=data.index
    )


    # ========================================================
    # Keep all columns needed for tooltip
    # ========================================================

    available_tooltip_columns = [
        column
        for column in tooltip_columns
        if column in data.columns
    ]


    # ========================================================
    # Final dataframe
    # ========================================================

    final_df = pd.concat(
        [
            data[
                available_tooltip_columns
            ],
            pca_df
        ],
        axis=1
    )


    # ========================================================
    # PCA loadings
    # ========================================================

    loadings = (
        pca.components_.T
        * np.sqrt(
            pca.explained_variance_
        )
    )


    loading_df = pd.DataFrame(
        loadings,
        columns=[
            "PC1",
            "PC2"
        ],
        index=pca_vars
    ).reset_index()


    loading_df.rename(
        columns={
            "index": "variable"
        },
        inplace=True
    )


    # ========================================================
    # Descriptive names for loading variables
    # ========================================================

    loading_df["variable_label"] = (
        loading_df["variable"]
        .map(
            lambda x: column_labels.get(
                x,
                x.replace(
                    "_",
                    " "
                ).title()
            )
        )
    )


    # ========================================================
    # Scale loading vectors
    # ========================================================

    max_pc = np.max(
        np.abs(
            final_df[
                [
                    "PC1",
                    "PC2"
                ]
            ].values
        )
    )


    max_loading = np.max(
        np.abs(
            loading_df[
                [
                    "PC1",
                    "PC2"
                ]
            ].values
        )
    )


    if max_loading == 0:

        vector_scale = 1

    else:

        vector_scale = (
            max_pc
            / max_loading
            * 0.7
        )


    loading_df["PC1_scaled"] = (
        loading_df["PC1"]
        * vector_scale
    )

    loading_df["PC2_scaled"] = (
        loading_df["PC2"]
        * vector_scale
    )


    # ========================================================
    # Vector dataframe
    # ========================================================

    vector_rows = []

    for _, row in loading_df.iterrows():

        vector_rows.append(
            {
                "x": 0,
                "y": 0,
                "x2": row["PC1_scaled"],
                "y2": row["PC2_scaled"],
                "variable": row["variable_label"]
            }
        )


    vector_df = pd.DataFrame(
        vector_rows
    )


    # ========================================================
    # Tooltip
    # ========================================================

    tooltip = [

        alt.Tooltip(
            column,
            title=column_labels.get(
                column,
                column.replace(
                    "_",
                    " "
                ).title()
            )
        )

        for column in tooltip_columns

        if column in final_df.columns

    ]


    # --------------------------------------------------------
    # Add PCA coordinates
    # --------------------------------------------------------

    tooltip += [

        alt.Tooltip(
            "PC1:Q",
            title="PC1",
            format=".3f"
        ),

        alt.Tooltip(
            "PC2:Q",
            title="PC2",
            format=".3f"
        )

    ]


    # ========================================================
    # PCA points
    # ========================================================

    points = (

        alt.Chart(
            final_df
        )

        .mark_circle(
            size=70
        )

        .encode(


            # ------------------------------------------------
            # X axis
            # ------------------------------------------------

            x=alt.X(
                "PC1:Q",
                title=(
                    f"PC1 "
                    f"({pca.explained_variance_ratio_[0] * 100:.2f}%)"
                )
            ),


            # ------------------------------------------------
            # Y axis
            # ------------------------------------------------

            y=alt.Y(
                "PC2:Q",
                title=(
                    f"PC2 "
                    f"({pca.explained_variance_ratio_[1] * 100:.2f}%)"
                )
            ),


            # ------------------------------------------------
            # Color
            # ------------------------------------------------

            color=alt.Color(

                "point_category:N",

                scale=alt.Scale(

                    domain=point_category_order,

                    range=point_category_colors

                ),

                legend=alt.Legend(
                    title=(
                        "Functional Group / "
                        "Special Samples"
                    )
                )
            ),


            # ------------------------------------------------
            # Tooltip
            # ------------------------------------------------

            tooltip=tooltip

        )
    )


    # ========================================================
    # Loading vectors
    # ========================================================

    vectors = (

        alt.Chart(
            vector_df
        )

        .mark_rule(
            color="red",
            opacity=0.7
        )

        .encode(

            x=alt.X(
                "x:Q"
            ),

            y=alt.Y(
                "y:Q"
            ),

            x2="x2:Q",

            y2="y2:Q"

        )
    )


    # ========================================================
    # Loading labels
    # ========================================================

    labels = (

        alt.Chart(
            vector_df
        )

        .mark_text(
            align="left",
            dx=5,
            dy=-5,
            color="darkred",
            fontWeight="bold",
            fontSize=12
        )

        .encode(

            x=alt.X(
                "x2:Q"
            ),

            y=alt.Y(
                "y2:Q"
            ),

            text=alt.Text(
                "variable:N"
            )

        )
    )


    # ========================================================
    # Final PCA biplot
    # ========================================================

    final_plot = (

        points
        + vectors
        + labels

    ).properties(

        width=850,

        height=600,

        title=(
            f"PCA Biplot - "
            f"{functional_group} "
            f"(n = {len(final_df)})"
        )

    ).configure_axis(

        labelFontSize=14,

        titleFontSize=18

    ).configure_legend(

        titleFontSize=16,

        labelFontSize=14

    ).configure_title(

        fontSize=20

    ).interactive()


    # ========================================================
    # Display
    # ========================================================

    display(
        final_plot
    )


# ============================================================
# Interactive output
# ============================================================

pca_output = widgets.interactive_output(
    plot_pca,
    {
        "functional_group":
            pca_functional_widget
    }
)

In [39]:
# ============================================================
# Requirements
# ============================================================
# pip install altair vl-convert-python pandas numpy scikit-learn
#
# vl-convert-python is required so that chart.to_json()-based
# rendering works cleanly.

import json
import numpy as np
import pandas as pd
import altair as alt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Disable Altair's default 5000-row safety limit, in case df_pca
# (times the number of functional groups) exceeds it.
alt.data_transformers.disable_max_rows()

# ============================================================
# PCA variables
# ============================================================

pca_vars = [
    "dm_percentage",
    "ash_dm",
    "om_percentage",
    "pc_percentage_dm",
    "adf_percentage_dm",
    "ndf_percentage_dm",
    "ch4_percentage_in_gas_8h",
    "ch4_percentage_in_gas_24h",
    "methane_intensity",
    "tddm"
]

# ============================================================
# Column labels
# ============================================================

column_labels = {
    "id": "Accession identifier",
    "id_lab": "Laboratory sample identifier",
    "tax_name": "Taxonomic name",
    "functional_group": "Functional group",
    "point_category": "Plot category",
    "subset": "Experimental subset",
    "n_replicates_nutrition": "No. replicates (Nutrition)",
    "dm_percentage": "Dry matter (%)",
    "ash_dm": "Ash (% DM)",
    "om_percentage": "Organic matter (% DM)",
    "pc_percentage_dm": "Crude protein (% DM)",
    "adf_percentage_dm": "ADF (% DM)",
    "ndf_percentage_dm": "NDF (% DM)",
    "n_replicates_gas": "No. replicates (Gas)",
    "ch4_percentage_in_gas_8h": "Methane in Gas (8 h, %)",
    "ch4_percentage_in_gas_24h": "Methane in Gas (24 h, %)",
    "methane_intensity": "Methane intensity",
    "tddm": "TDDM (%)",
    "ch4_category": "Methane category",
    "tddm_category": "TDDM category",
    "lmf_category": "LMF category",
    "lmf_category_rank": "Position on LMF category",
    "quartile": "LMF quartile",
    "quartile_rank": "LMF ranking"
}

tooltip_columns = [
    "id", "id_lab", "tax_name", "functional_group", "point_category",
    "subset", "n_replicates_nutrition", "dm_percentage", "ash_dm",
    "om_percentage", "pc_percentage_dm", "adf_percentage_dm",
    "ndf_percentage_dm", "n_replicates_gas", "ch4_percentage_in_gas_8h",
    "ch4_percentage_in_gas_24h", "methane_intensity", "tddm",
    "ch4_category", "tddm_category", "lmf_category",
    "lmf_category_rank", "quartile", "quartile_rank"
]

# "Target" is left out here on purpose: in the upstream df_pca-building
# code, the line that assigns point_category = "Target" is commented
# out, so no row ever actually takes that value. Keeping it in the
# scale's domain would leave a permanent, empty "Target" legend entry.
point_category_order = [
    "Grasses", "Herbaceous_legumes", "Shrub_Trees",
    "Benchmark", "Star grass control"
]

point_category_colors = [
    "#4C78A8", "#59A14F", "#F28E2B",
    "purple", "black"
]

# ============================================================
# NOTE: this assumes `df_pca` has already been built exactly as
# in the original script (filtered, with point_category assigned, etc.)
# ============================================================

missing_pca_vars = [c for c in pca_vars if c not in df_pca.columns]
if missing_pca_vars:
    raise ValueError(f"The following PCA variables are missing from df_pca: {missing_pca_vars}")

functional_groups = sorted(df_pca["functional_group"].dropna().unique().tolist())
groups_to_run = ["All"] + functional_groups


def compute_pca_for_group(data, group_name):
    """Run PCA on a subset of the data and return (points, vectors), both labeled."""

    if len(data) < 3:
        print(f"Not enough observations for '{group_name}'. Found {len(data)} samples.")
        return None, None

    X = data[pca_vars].copy()
    X.replace([np.inf, -np.inf], np.nan, inplace=True)
    X = X.fillna(X.mean())

    if X.isna().any().any():
        bad = X.columns[X.isna().any()].tolist()
        print(f"These variables still contain missing values for '{group_name}': {bad}")
        return None, None

    zero_var = [c for c in pca_vars if X[c].nunique() <= 1]
    if zero_var:
        print(f"These variables have zero variance for '{group_name}': {zero_var}")
        return None, None

    # Standardization
    X_scaled = StandardScaler().fit_transform(X)

    # PCA
    pca = PCA(n_components=2)
    pcs = pca.fit_transform(X_scaled)

    pc1_pct = pca.explained_variance_ratio_[0] * 100
    pc2_pct = pca.explained_variance_ratio_[1] * 100

    pca_df = pd.DataFrame(pcs, columns=["PC1", "PC2"], index=data.index)

    available_tooltip_cols = [c for c in tooltip_columns if c in data.columns]
    final_df = pd.concat([data[available_tooltip_cols], pca_df], axis=1)
    final_df["pca_run"] = group_name
    final_df["axis_title_x"] = f"PC1 ({pc1_pct:.2f}%)"
    final_df["axis_title_y"] = f"PC2 ({pc2_pct:.2f}%)"
    # Percentages are folded into the title too, as a guaranteed fallback
    # in case the JS-based axis-title update (below) can't find the
    # right DOM nodes in a given browser/Vega version.
    final_df["plot_title"] = (
        f"PCA Biplot - {group_name} (n = {len(final_df)})  "
        f"[PC1: {pc1_pct:.2f}%, PC2: {pc2_pct:.2f}%]"
    )

    # PCA loadings
    loadings = pca.components_.T * np.sqrt(pca.explained_variance_)
    loading_df = pd.DataFrame(loadings, columns=["PC1", "PC2"], index=pca_vars).reset_index()
    loading_df.rename(columns={"index": "variable"}, inplace=True)
    loading_df["variable_label"] = loading_df["variable"].map(
        lambda x: column_labels.get(x, x.replace("_", " ").title())
    )

    # Scale loading vectors
    max_pc = np.max(np.abs(final_df[["PC1", "PC2"]].values))
    max_loading = np.max(np.abs(loading_df[["PC1", "PC2"]].values))
    vector_scale = 1 if max_loading == 0 else (max_pc / max_loading * 0.7)

    loading_df["PC1_scaled"] = loading_df["PC1"] * vector_scale
    loading_df["PC2_scaled"] = loading_df["PC2"] * vector_scale

    vector_rows = []
    for _, row in loading_df.iterrows():
        vector_rows.append({
            "x": 0, "y": 0,
            "x2": row["PC1_scaled"], "y2": row["PC2_scaled"],
            "variable": row["variable_label"],
            "pca_run": group_name
        })
    vector_df = pd.DataFrame(vector_rows)

    return final_df, vector_df


# ============================================================
# Pre-compute PCA for every group
# ============================================================

all_points, all_vectors = [], []

for group in groups_to_run:
    subset = df_pca.copy() if group == "All" else df_pca[df_pca["functional_group"] == group].copy()
    pts, vecs = compute_pca_for_group(subset, group)
    if pts is not None:
        all_points.append(pts)
        all_vectors.append(vecs)

points_all = pd.concat(all_points, ignore_index=True)
vectors_all = pd.concat(all_vectors, ignore_index=True)

# Per-group axis title lookup, embedded into the HTML for the JS-based
# axis title update (best effort — see note in the JS section below).
axis_titles_lookup = (
    points_all
    .drop_duplicates("pca_run")
    .set_index("pca_run")[["axis_title_x", "axis_title_y"]]
    .rename(columns={"axis_title_x": "x", "axis_title_y": "y"})
    .to_dict(orient="index")
)

# ============================================================
# Interactive dropdown selector (rendered at the top of the chart)
# ============================================================

group_dropdown = alt.binding_select(
    options=groups_to_run,
    name="Functional group: "
)

group_param = alt.param(
    name="sel_group",
    value="All",
    bind=group_dropdown
)

group_filter_expr = "datum.pca_run == sel_group"

# ------------------------------------------------------------
# Text search box: highlight by id or id_lab, comma-separated,
# partial (substring) match, case-insensitive
# ------------------------------------------------------------

search_param = alt.param(
    name="search_id",
    value="",
    bind=alt.binding(input="text", name="Search ID or Lab ID (comma-separated): ")
)

search_pattern_expr = (
    "replace(replace(search_id, /\\s+/g, ''), /,/g, '|')"
)

is_match_expr = (
    f"search_id !== '' && ("
    f"test(regexp({search_pattern_expr}, 'i'), toString(datum.id)) || "
    f"test(regexp({search_pattern_expr}, 'i'), toString(datum.id_lab))"
    f")"
)

tooltip = [
    alt.Tooltip(col, title=column_labels.get(col, col.replace("_", " ").title()))
    for col in tooltip_columns
    if col in points_all.columns
] + [
    alt.Tooltip("PC1:Q", title="PC1", format=".3f"),
    alt.Tooltip("PC2:Q", title="PC2", format=".3f"),
]

# ------------------------------------------------------------
# Title row (includes the PC1/PC2 percentages as a guaranteed
# fallback, in addition to the best-effort axis title update)
# ------------------------------------------------------------

title_chart = (
    alt.Chart(points_all)
    .mark_text(align="center", fontSize=18, dy=0)
    .encode(text="plot_title:N")
    .transform_filter(group_filter_expr)
    .properties(width=850, height=30)
)

# ------------------------------------------------------------
# PCA points — base layer (all non-matched points, by category)
# and highlight layer (only matched points, one shape per id)
# ------------------------------------------------------------

base_layer = (
    alt.Chart(points_all)
    .mark_circle(size=70)
    .transform_filter(group_filter_expr)
    .transform_filter(f"!({is_match_expr})")
    .encode(
        x=alt.X("PC1:Q", title="PC1"),
        y=alt.Y("PC2:Q", title="PC2"),
        color=alt.Color(
            "point_category:N",
            scale=alt.Scale(domain=point_category_order, range=point_category_colors),
            legend=alt.Legend(title="Functional Group", symbolSize=70)
        ),
        tooltip=tooltip
    )
)

highlight_layer = (
    alt.Chart(points_all)
    .mark_point(filled=True, size=70, stroke="black", strokeWidth=1.5)
    .transform_filter(group_filter_expr)
    .transform_filter(is_match_expr)
    .encode(
        x=alt.X("PC1:Q"),
        y=alt.Y("PC2:Q"),
        # Only matched rows are ever present in this layer's data, so the
        # shape scale's domain — and therefore the legend — automatically
        # lists exactly the searched accession(s), and only those.
        shape=alt.Shape(
            "id:N",
            legend=alt.Legend(title="Highlighted accession(s)", symbolSize=70, titleLimit=0)
        ),
        color=alt.value("red"),
        tooltip=tooltip
    )
    .add_params(search_param)
)

points_chart = base_layer + highlight_layer

# ------------------------------------------------------------
# Loading vectors
# ------------------------------------------------------------

vectors_chart = (
    alt.Chart(vectors_all)
    .mark_rule(color="red", opacity=0.7)
    .encode(x="x:Q", y="y:Q", x2="x2:Q", y2="y2:Q")
    .transform_filter(group_filter_expr)
)

# ------------------------------------------------------------
# Loading labels
# ------------------------------------------------------------

labels_chart = (
    alt.Chart(vectors_all)
    .mark_text(align="left", dx=5, dy=-5, color="darkred", fontWeight="bold", fontSize=12)
    .encode(x="x2:Q", y="y2:Q", text="variable:N")
    .transform_filter(group_filter_expr)
)

# ------------------------------------------------------------
# Final PCA biplot — dropdown on top, then title, then the plot
# ------------------------------------------------------------

biplot = (
    (points_chart + vectors_chart + labels_chart)
    .properties(width=850, height=600)
)

final_plot = (
    alt.vconcat(title_chart, biplot)
    .add_params(group_param, search_param)
    .configure_axis(labelFontSize=14, titleFontSize=18)
    .configure_legend(titleFontSize=16, labelFontSize=14)
    .configure_view(strokeWidth=0)
    .interactive()
)

# ============================================================
# Save as interactive HTML with the dropdown forced above the chart
# ============================================================

chart_spec = final_plot.to_json(indent=None)
axis_titles_json = json.dumps(axis_titles_lookup)

html_template = f"""
<!DOCTYPE html>
<html>
<head>
  <meta charset="utf-8" />
  <title>PCA Biplot</title>
  <script src="https://cdn.jsdelivr.net/npm/vega@5"></script>
  <script src="https://cdn.jsdelivr.net/npm/vega-lite@5"></script>
  <script src="https://cdn.jsdelivr.net/npm/vega-embed@6"></script>
  <script src="https://cdn.jsdelivr.net/npm/utif@3.1.0/UTIF.js"></script>
  <script src="https://cdnjs.cloudflare.com/ajax/libs/jspdf/2.5.1/jspdf.umd.min.js"></script>
  <style>
    body {{
      font-family: sans-serif;
      display: flex;
      flex-direction: column;
      align-items: center;
    }}
    #controls {{
      margin: 20px 0;
      font-size: 16px;
    }}
    #download-buttons {{
      margin: 20px 0 10px 0;
      display: flex;
      align-items: center;
      gap: 10px;
    }}
    #download-buttons select,
    #download-buttons button {{
      padding: 8px 16px;
      font-size: 14px;
      border: 1px solid #ccc;
      border-radius: 6px;
      background: #f7f7f9;
      cursor: pointer;
    }}
    #download-buttons button:hover {{
      background: #e9e9ee;
    }}
  </style>
</head>
<body>
  <div id="controls"></div>
  <div id="vis"></div>
  <div id="download-buttons">
    <select id="format-select">
      <option value="png">PNG</option>
      <option value="jpg">JPG</option>
      <option value="tiff">TIFF</option>
      <option value="pdf">PDF</option>
    </select>
    <button id="btn-download">Download</button>
  </div>

  <script type="text/javascript">
    const spec = {chart_spec};
    const axisTitles = {axis_titles_json};

    // Vega renders at 96 CSS pixels per inch by default. Scaling the
    // canvas by (300 / 96) makes the exported pixel dimensions match
    // a 300 dpi print at the chart's original physical size, for
    // every format below (PNG, JPG, TIFF, and PDF alike).
    const DPI = 300;
    const DPI_SCALE = DPI / 96;

    vegaEmbed("#vis", spec, {{
      actions: false,
      bindingsElement: "#controls",
      renderer: "svg"
    }}).then(function(result) {{

      const view = result.view;

      function triggerDownload(url, filename) {{
        const link = document.createElement("a");
        link.href = url;
        link.download = filename;
        document.body.appendChild(link);
        link.click();
        document.body.removeChild(link);
      }}

      function downloadAs(format) {{
        view.toCanvas(DPI_SCALE).then(function(canvas) {{

          if (format === "png") {{
            triggerDownload(canvas.toDataURL("image/png"), "pca_plot.png");

          }} else if (format === "jpg") {{
            triggerDownload(canvas.toDataURL("image/jpeg", 0.95), "pca_plot.jpg");

          }} else if (format === "tiff") {{
            const ctx = canvas.getContext("2d");
            const imageData = ctx.getImageData(0, 0, canvas.width, canvas.height);
            const tiffBuffer = UTIF.encodeImage(imageData.data, canvas.width, canvas.height);
            const blob = new Blob([tiffBuffer], {{ type: "image/tiff" }});
            const url = URL.createObjectURL(blob);
            triggerDownload(url, "pca_plot.tiff");

          }} else if (format === "pdf") {{
            const widthIn = canvas.width / DPI;
            const heightIn = canvas.height / DPI;
            const {{ jsPDF }} = window.jspdf;
            const pdf = new jsPDF({{
              orientation: widthIn >= heightIn ? "landscape" : "portrait",
              unit: "in",
              format: [widthIn, heightIn]
            }});
            pdf.addImage(canvas.toDataURL("image/png"), "PNG", 0, 0, widthIn, heightIn);
            pdf.save("pca_plot.pdf");
          }}

        }}).catch(console.error);
      }}

      document.getElementById("btn-download").addEventListener("click", function() {{
        const format = document.getElementById("format-select").value;
        downloadAs(format);
      }});

      // Explicitly hide the "Highlighted accession(s)" legend while the
      // search box is empty (Vega-Lite already leaves it empty of
      // entries in that case, but this makes it fully invisible).
      function toggleHighlightedAccessionsLegend(searchValue) {{
        const container = view.container();
        if (!container) return;

        const isActive = !!(searchValue && searchValue.trim() !== "");

        const legendGroups = container.querySelectorAll('[aria-roledescription="legend"]');
        legendGroups.forEach(function(g) {{
          const texts = g.querySelectorAll("text");
          let isTargetLegend = false;
          texts.forEach(function(t) {{
            if (t.textContent.trim().indexOf("Highlighted") === 0) {{
              isTargetLegend = true;
            }}
          }});
          if (isTargetLegend) {{
            g.style.display = isActive ? "" : "none";
          }}
        }});
      }}

      // BEST EFFORT: update the actual x/y axis titles to show the
      // PC1/PC2 percentages for the currently selected group. This
      // inspects the rendered SVG directly (Vega-Lite has no built-in
      // way to bind an axis title to a live parameter), so it depends
      // on internal DOM structure that could vary between Vega
      // versions. The percentages are ALSO shown in the chart's main
      // title above, which does not depend on this and always works.
      function updateAxisTitles(group) {{
        const info = axisTitles[group];
        if (!info) return;
        const container = view.container();
        if (!container) return;

        const axisGroups = container.querySelectorAll('[aria-roledescription="axis"]');
        axisGroups.forEach(function(g) {{
          // The axis title is typically a direct <text> child of the
          // axis group, while tick labels live inside a nested <g>.
          const titleEl = g.querySelector(":scope > text");
          if (!titleEl) return;
          const transform = titleEl.getAttribute("transform") || "";
          const isRotated = transform.indexOf("rotate") !== -1;
          titleEl.textContent = isRotated ? info.y : info.x;
        }});
      }}

      function refreshDynamicLabels() {{
        const group = view.signal("sel_group") || "All";
        updateAxisTitles(group);
      }}

      toggleHighlightedAccessionsLegend(view.signal("search_id"));
      view.addSignalListener("search_id", function(name, value) {{
        setTimeout(function() {{ toggleHighlightedAccessionsLegend(value); }}, 50);
      }});

      // Re-apply axis titles once on load, and whenever the dropdown
      // selection changes.
      setTimeout(refreshDynamicLabels, 50);
      view.addSignalListener("sel_group", function() {{
        setTimeout(refreshDynamicLabels, 50);
      }});

    }}).catch(console.error);
  </script>
</body>
</html>
"""

with open("pca.html", "w", encoding="utf-8") as f:
    f.write(html_template)

print("Done: pca.html")

Done: pca.html


In [40]:
with open("/content/drive/MyDrive/lmf/output/2026_08_31_web_visualizer/pca.html", "w", encoding="utf-8") as f:
    f.write(html_template)

## 7.3 UMAP

In [41]:
# ============================================================
# UMAP variables
# ============================================================

umap_vars = [
    "dm_percentage",
    "ash_dm",
    "om_percentage",
    "pc_percentage_dm",
    "adf_percentage_dm",
    "ndf_percentage_dm",
    "ch4_percentage_in_gas_8h",
    "ch4_percentage_in_gas_24h",
    "methane_intensity",
    "tddm"
]


# ============================================================
# Column labels
# ============================================================

column_labels = {
    "id": "Accession identifier",
    "id_lab": "Laboratory sample identifier",
    "tax_name": "Taxonomic name",
    "functional_group": "Functional group",
    "subset": "Experimental subset",
    "n_replicates_nutrition": "No. replicates (Nutrition)",
    "dm_percentage": "Dry matter (%)",
    "ash_dm": "Ash (% DM)",
    "om_percentage": "Organic matter (% DM)",
    "pc_percentage_dm": "Crude protein (% DM)",
    "adf_percentage_dm": "ADF (% DM)",
    "ndf_percentage_dm": "NDF (% DM)",
    "n_replicates_gas": "No. replicates (Gas)",
    "ch4_percentage_in_gas_8h": "Methane in Gas (8 h, %)",
    "ch4_percentage_in_gas_24h": "Methane in Gas (24 h, %)",
    "methane_intensity": "Methane intensity",
    "tddm": "TDDM (%)",
    "ch4_category": "Methane category",
    "tddm_category": "TDDM category",
    "lmf_category": "LMF category",
    "lmf_category_rank": "Position on LMF category",
    "quartile": "LMF quartile",
    "quartile_rank": "LMF ranking"
}


# ============================================================
# Tooltip columns
# ============================================================

tooltip_columns = [
    "id",
    "id_lab",
    "tax_name",
    "functional_group",
    "subset",
    "n_replicates_nutrition",
    "dm_percentage",
    "ash_dm",
    "om_percentage",
    "pc_percentage_dm",
    "adf_percentage_dm",
    "ndf_percentage_dm",
    "n_replicates_gas",
    "ch4_percentage_in_gas_8h",
    "ch4_percentage_in_gas_24h",
    "methane_intensity",
    "tddm",
    "ch4_category",
    "tddm_category",
    "lmf_category",
    "lmf_category_rank",
    "quartile",
    "quartile_rank"
]


# ============================================================
# Check UMAP variables
# ============================================================

missing_umap_vars = [
    column
    for column in umap_vars
    if column not in df.columns
]

if missing_umap_vars:

    raise ValueError(
        "The following UMAP variables are missing from df: "
        f"{missing_umap_vars}"
    )


# ============================================================
# Clean functional group
# ============================================================

df["functional_group"] = (
    df["functional_group"]
    .astype("string")
    .str.strip()
)


# ============================================================
# Valid functional groups
# ============================================================

valid_functional_groups = [
    "Grasses",
    "Herbaceous_legumes",
    "Shrub_Trees"
]


# ============================================================
# Filter data for UMAP
# ============================================================

df_umap = df[
    df["functional_group"].isin(
        valid_functional_groups
    )
].copy()


# ============================================================
# Special sample categories
# ============================================================

# Default category = functional group
df_umap["point_category"] = (
    df_umap["functional_group"]
)


# ------------------------------------------------------------
# Benchmark samples
# ------------------------------------------------------------

df_umap.loc[
    df_umap["id"].isin(benchmark_ids),
    "point_category"
] = "Benchmark"


# ------------------------------------------------------------
# Star grass control
# ------------------------------------------------------------

df_umap.loc[
    df_umap["id"].isin(star_grass_control_ids),
    "point_category"
] = "Star grass control"


# ------------------------------------------------------------
# Target sample
# ------------------------------------------------------------

df_umap.loc[
    df_umap["id"].isin(target),
    "point_category"
] = "Target"


# ============================================================
# Functional groups available for widget
# ============================================================

functional_groups = sorted(
    df_umap["functional_group"]
    .dropna()
    .unique()
    .tolist()
)


# ============================================================
# Functional group widget
# ============================================================

umap_functional_widget = widgets.Dropdown(
    options=["All"] + functional_groups,
    value="All",
    description="Group:"
)


# ============================================================
# UMAP parameters
# ============================================================

n_neighbors = 15
min_dist = 0.1
random_state = 42


# ============================================================
# Point category order
# ============================================================

point_category_order = [
    "Grasses",
    "Herbaceous_legumes",
    "Shrub_Trees",
    "Benchmark",
    "Star grass control",
    "Target"
]


# ============================================================
# Point category colors
# ============================================================

point_category_colors = [
    "#4C78A8",   # Grasses
    "#59A14F",   # Herbaceous legumes
    "#F28E2B",   # Shrub/Trees
    "purple",   # Benchmark
    "black",   # Star grass control
    "red"    # Target
]


# ============================================================
# UMAP function
# ============================================================

def plot_umap(functional_group):


    # ========================================================
    # Filter data by functional group
    # ========================================================

    if functional_group == "All":

        data = df_umap.copy()

    else:

        data = df_umap[
            df_umap["functional_group"]
            == functional_group
        ].copy()


    # ========================================================
    # Check number of observations
    # ========================================================

    if len(data) < 3:

        print(
            f"Not enough observations for "
            f"'{functional_group}'. "
            f"Found {len(data)} samples."
        )

        return


    # ========================================================
    # UMAP matrix
    # ========================================================

    X = data[umap_vars].copy()


    # ========================================================
    # Replace infinite values with NaN
    # ========================================================

    X.replace(
        [np.inf, -np.inf],
        np.nan,
        inplace=True
    )


    # ========================================================
    # Impute missing values with column means
    # ========================================================

    X = X.fillna(
        X.mean()
    )


    # ========================================================
    # Check remaining NaNs
    # ========================================================

    if X.isna().any().any():

        bad_columns = X.columns[
            X.isna().any()
        ].tolist()

        print(
            "These variables still contain missing values: "
            f"{bad_columns}"
        )

        return


    # ========================================================
    # Check zero variance
    # ========================================================

    zero_variance = [
        column
        for column in umap_vars
        if X[column].nunique() <= 1
    ]

    if zero_variance:

        print(
            "These variables have zero variance for "
            f"'{functional_group}': "
            f"{zero_variance}"
        )

        return


    # ========================================================
    # Standardization
    # ========================================================

    scaler = StandardScaler()

    X_scaled = scaler.fit_transform(
        X
    )


    # ========================================================
    # Adjust n_neighbors if necessary
    # ========================================================

    effective_n_neighbors = min(
        n_neighbors,
        len(X) - 1
    )


    # ========================================================
    # UMAP
    # ========================================================

    reducer = umap.UMAP(
        n_components=2,
        n_neighbors=effective_n_neighbors,
        min_dist=min_dist,
        metric="euclidean",
        random_state=random_state
    )


    embedding = reducer.fit_transform(
        X_scaled
    )


    # ========================================================
    # UMAP coordinates
    # ========================================================

    umap_df = pd.DataFrame(
        embedding,
        columns=[
            "UMAP1",
            "UMAP2"
        ],
        index=data.index
    )


    # ========================================================
    # Keep columns needed for tooltip
    # ========================================================

    available_tooltip_columns = [
        column
        for column in tooltip_columns
        if column in data.columns
    ]


    # ========================================================
    # Final dataframe
    # ========================================================

    final_df = pd.concat(
        [
            data[
                available_tooltip_columns
                + ["point_category"]
            ],
            umap_df
        ],
        axis=1
    )


    # ========================================================
    # Tooltip
    # ========================================================

    tooltip = [

        alt.Tooltip(
            column,
            title=column_labels.get(
                column,
                column.replace(
                    "_",
                    " "
                ).title()
            )
        )

        for column in tooltip_columns

        if column in final_df.columns

    ]


    # --------------------------------------------------------
    # Add UMAP coordinates to tooltip
    # --------------------------------------------------------

    tooltip += [

        alt.Tooltip(
            "UMAP1:Q",
            title="UMAP1",
            format=".3f"
        ),

        alt.Tooltip(
            "UMAP2:Q",
            title="UMAP2",
            format=".3f"
        )

    ]


    # ========================================================
    # UMAP points
    # ========================================================

    points = (

        alt.Chart(
            final_df
        )

        .mark_circle(
            size=70
        )

        .encode(


            # ------------------------------------------------
            # X axis
            # ------------------------------------------------

            x=alt.X(
                "UMAP1:Q",
                title="UMAP 1"
            ),


            # ------------------------------------------------
            # Y axis
            # ------------------------------------------------

            y=alt.Y(
                "UMAP2:Q",
                title="UMAP 2"
            ),


            # ------------------------------------------------
            # Color
            # ------------------------------------------------

            color=alt.Color(

                "point_category:N",

                scale=alt.Scale(

                    domain=point_category_order,

                    range=point_category_colors

                ),

                legend=alt.Legend(
                    title=(
                        "Functional Group / "
                        "Special Samples"
                    )
                )
            ),


            # ------------------------------------------------
            # Tooltip
            # ------------------------------------------------

            tooltip=tooltip

        )
    )


    # ========================================================
    # Final UMAP plot
    # ========================================================

    final_plot = (

        points

        .properties(

            width=850,

            height=600,

            title=(
                f"UMAP — "
                f"{functional_group} "
                f"(n = {len(final_df)})"
            )

        )

        .configure_axis(

            labelFontSize=14,

            titleFontSize=18

        )

        .configure_legend(

            titleFontSize=16,

            labelFontSize=14

        )

        .configure_title(

            fontSize=20

        )

        .interactive()

    )


    # ========================================================
    # Display
    # ========================================================

    display(
        final_plot
    )


# ============================================================
# Interactive output
# ============================================================

umap_output = widgets.interactive_output(

    plot_umap,

    {
        "functional_group":
            umap_functional_widget
    }

)




In [42]:
# ============================================================
# Requirements
# ============================================================
# pip install altair vl-convert-python pandas numpy scikit-learn umap-learn
#
# vl-convert-python is required so that chart.to_json() based
# rendering works cleanly; umap-learn provides the UMAP() reducer.

import numpy as np
import pandas as pd
import altair as alt
import umap
from sklearn.preprocessing import StandardScaler

# Disable Altair's default 5000-row safety limit, in case df_umap
# (times the number of functional groups) exceeds it.
alt.data_transformers.disable_max_rows()

# ============================================================
# UMAP variables
# ============================================================

umap_vars = [
    "dm_percentage",
    "ash_dm",
    "om_percentage",
    "pc_percentage_dm",
    "adf_percentage_dm",
    "ndf_percentage_dm",
    "ch4_percentage_in_gas_8h",
    "ch4_percentage_in_gas_24h",
    "methane_intensity",
    "tddm"
]

# ============================================================
# Column labels
# ============================================================

column_labels = {
    "id": "Accession identifier",
    "id_lab": "Laboratory sample identifier",
    "tax_name": "Taxonomic name",
    "functional_group": "Functional group",
    "subset": "Experimental subset",
    "n_replicates_nutrition": "No. replicates (Nutrition)",
    "dm_percentage": "Dry matter (%)",
    "ash_dm": "Ash (% DM)",
    "om_percentage": "Organic matter (% DM)",
    "pc_percentage_dm": "Crude protein (% DM)",
    "adf_percentage_dm": "ADF (% DM)",
    "ndf_percentage_dm": "NDF (% DM)",
    "n_replicates_gas": "No. replicates (Gas)",
    "ch4_percentage_in_gas_8h": "Methane in Gas (8 h, %)",
    "ch4_percentage_in_gas_24h": "Methane in Gas (24 h, %)",
    "methane_intensity": "Methane intensity",
    "tddm": "TDDM (%)",
    "ch4_category": "Methane category",
    "tddm_category": "TDDM category",
    "lmf_category": "LMF category",
    "lmf_category_rank": "Position on LMF category",
    "quartile": "LMF quartile",
    "quartile_rank": "LMF ranking"
}

tooltip_columns = [
    "id", "id_lab", "tax_name", "functional_group",
    "subset", "n_replicates_nutrition", "dm_percentage", "ash_dm",
    "om_percentage", "pc_percentage_dm", "adf_percentage_dm",
    "ndf_percentage_dm", "n_replicates_gas", "ch4_percentage_in_gas_8h",
    "ch4_percentage_in_gas_24h", "methane_intensity", "tddm",
    "ch4_category", "tddm_category", "lmf_category",
    "lmf_category_rank", "quartile", "quartile_rank"
]

point_category_order = [
    "Grasses", "Herbaceous_legumes", "Shrub_Trees",
    "Benchmark", "Star grass control"
]

point_category_colors = [
    "#4C78A8", "#59A14F", "#F28E2B",
    "purple", "black"
]

# ============================================================
# UMAP parameters
# ============================================================

n_neighbors = 15
min_dist = 0.1
random_state = 42

# ============================================================
# NOTE: this assumes `df_umap` has already been built exactly as
# in the original script (filtered, with point_category assigned, etc.)
# ============================================================

missing_umap_vars = [c for c in umap_vars if c not in df_umap.columns]
if missing_umap_vars:
    raise ValueError(f"The following UMAP variables are missing from df_umap: {missing_umap_vars}")

functional_groups = sorted(df_umap["functional_group"].dropna().unique().tolist())
groups_to_run = ["All"] + functional_groups


def compute_umap_for_group(data, group_name):
    """Run UMAP on a subset of the data and return the labeled embedding."""

    if len(data) < 3:
        print(f"Not enough observations for '{group_name}'. Found {len(data)} samples.")
        return None

    X = data[umap_vars].copy()
    X.replace([np.inf, -np.inf], np.nan, inplace=True)
    X = X.fillna(X.mean())

    if X.isna().any().any():
        bad = X.columns[X.isna().any()].tolist()
        print(f"These variables still contain missing values for '{group_name}': {bad}")
        return None

    zero_var = [c for c in umap_vars if X[c].nunique() <= 1]
    if zero_var:
        print(f"These variables have zero variance for '{group_name}': {zero_var}")
        return None

    # Standardization
    X_scaled = StandardScaler().fit_transform(X)

    # Adjust n_neighbors if necessary
    effective_n_neighbors = min(n_neighbors, len(X) - 1)

    # UMAP
    reducer = umap.UMAP(
        n_components=2,
        n_neighbors=effective_n_neighbors,
        min_dist=min_dist,
        metric="euclidean",
        random_state=random_state
    )
    embedding = reducer.fit_transform(X_scaled)

    umap_df = pd.DataFrame(embedding, columns=["UMAP1", "UMAP2"], index=data.index)

    available_tooltip_cols = [c for c in tooltip_columns if c in data.columns]
    final_df = pd.concat([data[available_tooltip_cols + ["point_category"]], umap_df], axis=1)
    final_df["umap_run"] = group_name
    final_df["plot_title"] = f"UMAP — {group_name} (n = {len(final_df)})"

    return final_df


# ============================================================
# Pre-compute UMAP for every group
# ============================================================

all_points = []

for group in groups_to_run:
    subset = df_umap.copy() if group == "All" else df_umap[df_umap["functional_group"] == group].copy()
    pts = compute_umap_for_group(subset, group)
    if pts is not None:
        all_points.append(pts)

points_all = pd.concat(all_points, ignore_index=True)

# ============================================================
# Interactive group dropdown
# ============================================================

group_dropdown = alt.binding_select(
    options=groups_to_run,
    name="Functional group: "
)

group_param = alt.selection_point(
    fields=["umap_run"],
    bind=group_dropdown,
    value=[{"umap_run": "All"}]
)

# ------------------------------------------------------------
# Text search box: highlight by id or id_lab, comma-separated,
# partial (substring) match, case-insensitive
# ------------------------------------------------------------

search_param = alt.param(
    name="search_id",
    value="",
    bind=alt.binding(input="text", name="Search ID or Lab ID (comma-separated): ")
)

search_pattern_expr = (
    "replace(replace(search_id, /\\s+/g, ''), /,/g, '|')"
)

is_match_expr = (
    f"search_id !== '' && ("
    f"test(regexp({search_pattern_expr}, 'i'), toString(datum.id)) || "
    f"test(regexp({search_pattern_expr}, 'i'), toString(datum.id_lab))"
    f")"
)

tooltip = [
    alt.Tooltip(col, title=column_labels.get(col, col.replace("_", " ").title()))
    for col in tooltip_columns
    if col in points_all.columns
] + [
    alt.Tooltip("UMAP1:Q", title="UMAP1", format=".3f"),
    alt.Tooltip("UMAP2:Q", title="UMAP2", format=".3f"),
]

# ------------------------------------------------------------
# Title row
# ------------------------------------------------------------

title_chart = (
    alt.Chart(points_all)
    .mark_text(align="center", fontSize=20, dy=0)
    .encode(text="plot_title:N")
    .transform_filter(group_param)
    .properties(width=850, height=30)
)

# ------------------------------------------------------------
# UMAP points
# ------------------------------------------------------------

base_layer = (
    alt.Chart(points_all)
    .mark_circle(size=70)
    .transform_calculate(
        # 'Target' is no longer a category in the color scale at all —
        # these rows just take their functional_group's normal color.
        display_category="datum.point_category === 'Target' ? datum.functional_group : datum.point_category"
    )
    .encode(
        x=alt.X("UMAP1:Q", title="UMAP 1"),
        y=alt.Y("UMAP2:Q", title="UMAP 2"),
        color=alt.Color(
            "display_category:N",
            scale=alt.Scale(domain=point_category_order, range=point_category_colors),
            legend=alt.Legend(title="Functional Group", symbolSize=70)
        ),
        tooltip=tooltip
    )
    .transform_filter(group_param)
    # Skip drawing matched points here — the highlight layer draws them
    # on top with a distinct shape instead.
    .transform_filter(f"!({is_match_expr})")
)

highlight_layer = (
    alt.Chart(points_all)
    .mark_point(filled=True, size=70, stroke="black", strokeWidth=1.5)
    .encode(
        x=alt.X("UMAP1:Q"),
        y=alt.Y("UMAP2:Q"),
        # Only matched rows are ever present in this layer's data, so the
        # shape scale's domain — and therefore the legend — automatically
        # lists exactly the searched accession(s), and only those.
        shape=alt.Shape(
            "id:N",
            legend=alt.Legend(title="Highlighted accession(s)", symbolSize=70, titleLimit=0)
        ),
        color=alt.value("red"),
        tooltip=tooltip
    )
    .transform_filter(group_param)
    .transform_filter(is_match_expr)
    .add_params(search_param)
)

points_chart = base_layer + highlight_layer

# ------------------------------------------------------------
# Final UMAP plot — dropdown + search box on top, then title, then plot
# ------------------------------------------------------------

biplot = points_chart.properties(width=850, height=600)

final_plot = (
    alt.vconcat(title_chart, biplot)
    .add_params(group_param)
    .configure_axis(labelFontSize=14, titleFontSize=18)
    .configure_legend(titleFontSize=16, labelFontSize=14)
    .configure_view(strokeWidth=0)
    .interactive()
)

# ============================================================
# Save as interactive HTML with controls forced above the chart
# ============================================================

chart_spec = final_plot.to_json(indent=None)

html_template = f"""
<!DOCTYPE html>
<html>
<head>
  <meta charset="utf-8" />
  <title>UMAP Plot</title>
  <script src="https://cdn.jsdelivr.net/npm/vega@5"></script>
  <script src="https://cdn.jsdelivr.net/npm/vega-lite@5"></script>
  <script src="https://cdn.jsdelivr.net/npm/vega-embed@6"></script>
  <script src="https://cdn.jsdelivr.net/npm/utif@3.1.0/UTIF.js"></script>
  <script src="https://cdnjs.cloudflare.com/ajax/libs/jspdf/2.5.1/jspdf.umd.min.js"></script>
  <style>
    body {{
      font-family: sans-serif;
      display: flex;
      flex-direction: column;
      align-items: center;
    }}
    #controls {{
      margin: 20px 0;
      font-size: 16px;
    }}
    #download-buttons {{
      margin: 20px 0 10px 0;
      display: flex;
      align-items: center;
      gap: 10px;
    }}
    #download-buttons select,
    #download-buttons button {{
      padding: 8px 16px;
      font-size: 14px;
      border: 1px solid #ccc;
      border-radius: 6px;
      background: #f7f7f9;
      cursor: pointer;
    }}
    #download-buttons button:hover {{
      background: #e9e9ee;
    }}
  </style>
</head>
<body>
  <div id="controls"></div>
  <div id="vis"></div>
  <div id="download-buttons">
    <select id="format-select">
      <option value="png">PNG</option>
      <option value="jpg">JPG</option>
      <option value="tiff">TIFF</option>
      <option value="pdf">PDF</option>
    </select>
    <button id="btn-download">Download</button>
  </div>

  <script type="text/javascript">
    const spec = {chart_spec};

    // Vega renders at 96 CSS pixels per inch by default. Scaling the
    // canvas by (300 / 96) makes the exported pixel dimensions match
    // a 300 dpi print at the chart's original physical size, for
    // every format below (PNG, JPG, TIFF, and PDF alike).
    const DPI = 300;
    const DPI_SCALE = DPI / 96;

    vegaEmbed("#vis", spec, {{
      actions: false,
      bindingsElement: "#controls",
      renderer: "svg"
    }}).then(function(result) {{

      const view = result.view;

      function triggerDownload(url, filename) {{
        const link = document.createElement("a");
        link.href = url;
        link.download = filename;
        document.body.appendChild(link);
        link.click();
        document.body.removeChild(link);
      }}

      function downloadAs(format) {{
        view.toCanvas(DPI_SCALE).then(function(canvas) {{

          if (format === "png") {{
            triggerDownload(canvas.toDataURL("image/png"), "umap_plot.png");

          }} else if (format === "jpg") {{
            triggerDownload(canvas.toDataURL("image/jpeg", 0.95), "umap_plot.jpg");

          }} else if (format === "tiff") {{
            const ctx = canvas.getContext("2d");
            const imageData = ctx.getImageData(0, 0, canvas.width, canvas.height);
            const tiffBuffer = UTIF.encodeImage(imageData.data, canvas.width, canvas.height);
            const blob = new Blob([tiffBuffer], {{ type: "image/tiff" }});
            const url = URL.createObjectURL(blob);
            triggerDownload(url, "umap_plot.tiff");

          }} else if (format === "pdf") {{
            // Page size (in inches) = pixel dimensions / DPI, so the
            // embedded image lands at exactly 300 dpi on the page.
            const widthIn = canvas.width / DPI;
            const heightIn = canvas.height / DPI;
            const {{ jsPDF }} = window.jspdf;
            const pdf = new jsPDF({{
              orientation: widthIn >= heightIn ? "landscape" : "portrait",
              unit: "in",
              format: [widthIn, heightIn]
            }});
            pdf.addImage(canvas.toDataURL("image/png"), "PNG", 0, 0, widthIn, heightIn);
            pdf.save("umap_plot.pdf");
          }}

        }}).catch(console.error);
      }}

      document.getElementById("btn-download").addEventListener("click", function() {{
        const format = document.getElementById("format-select").value;
        downloadAs(format);
      }});

      // Explicitly hide the "Highlighted accession(s)" legend while the
      // search box is empty (on top of Vega-Lite's own behavior of not
      // populating it when the shape scale's domain has no data).
      function toggleSelectedAccessionsLegend(searchValue) {{
        const container = view.container();
        if (!container) return;

        const isActive = !!(searchValue && searchValue.trim() !== "");

        // Vega tags each legend group with aria-roledescription="legend"
        // for accessibility — this is more stable across versions than
        // relying on internal CSS class names.
        const legendGroups = container.querySelectorAll('[aria-roledescription="legend"]');

        legendGroups.forEach(function(g) {{
          const texts = g.querySelectorAll("text");
          let isTargetLegend = false;
          texts.forEach(function(t) {{
            if (t.textContent.trim().indexOf("Highlighted") === 0) {{
              isTargetLegend = true;
            }}
          }});
          if (isTargetLegend) {{
            g.style.display = isActive ? "" : "none";
          }}
        }});
      }}

      toggleSelectedAccessionsLegend(view.signal("search_id"));
      view.addSignalListener("search_id", function(name, value) {{
        // Small delay: the signal fires slightly before Vega finishes
        // redrawing the SVG (e.g. adding/removing the legend group).
        setTimeout(function() {{ toggleSelectedAccessionsLegend(value); }}, 50);
      }});

    }}).catch(console.error);
  </script>
</body>
</html>
"""

with open("umap.html", "w", encoding="utf-8") as f:
    f.write(html_template)

print("Done: umap.html")

/usr/local/lib/python3.13/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/usr/local/lib/python3.13/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/usr/local/lib/python3.13/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/usr/local/lib/python3.13/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Done: interactive_umap.html


In [43]:
with open("/content/drive/MyDrive/lmf/output/2026_08_31_web_visualizer/umap.html", "w", encoding="utf-8") as f:
    f.write(html_template)

##  7.4 DBSCAN + UMAP

In [46]:
# ============================================================
# Lists of special samples
# ============================================================

target = [
    "CIAT-PM-21-2580"
]

star_grass_control_ids = [
    "Sample-8"
]

benchmark_ids = [
    "CIAT-6962-Mombaza-Ex",
    "CIAT-606-Basilisk-Opt",
    "CIAT-6294-Marandu-Opt",
    "CIAT-36087-MulatoII-Opt",
    "CIAT-6133-Llanero-Opt",
    "CIAT-6962-Mombaza-Opt",
    "Paja-Sabana-Madura",
    "Humidicola-679",
    "Sabana-Quemada",
    "CIAT-6294-Marandu-Exc",
    "CIAT-606-Basilisk-Exc",
    "BR-02-1752-Cayman-Exc",
    "BR-06-423-Cayman-Exc"
]


In [49]:
# ============================================================
# IMPORTS
# ============================================================

import numpy as np
import pandas as pd
import altair as alt
import umap.umap_ as umap

import ipywidgets as widgets

from IPython.display import display

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN


# ============================================================
# DBSCAN VARIABLES
# ============================================================

dbscan_vars = [
    "dm_percentage",
    "ash_dm",
    "om_percentage",
    "pc_percentage_dm",
    "adf_percentage_dm",
    "ndf_percentage_dm",
    "ch4_percentage_in_gas_8h",
    "ch4_percentage_in_gas_24h",
    "methane_intensity",
    "tddm"
]


# ============================================================
# COLUMN LABELS
# ============================================================

column_labels = {
    "id": "Accession identifier",
    "id_lab": "Laboratory sample identifier",
    "tax_name": "Taxonomic name",
    "functional_group": "Functional group",
    "subset": "Experimental subset",
    "n_replicates_nutrition": "No. replicates (Nutrition)",
    "dm_percentage": "Dry matter (%)",
    "ash_dm": "Ash (% DM)",
    "om_percentage": "Organic matter (% DM)",
    "pc_percentage_dm": "Crude protein (% DM)",
    "adf_percentage_dm": "ADF (% DM)",
    "ndf_percentage_dm": "NDF (% DM)",
    "n_replicates_gas": "No. replicates (Gas)",
    "ch4_percentage_in_gas_8h": "Methane in Gas (8 h, %)",
    "ch4_percentage_in_gas_24h": "Methane in Gas (24 h, %)",
    "methane_intensity": "Methane intensity",
    "tddm": "TDDM (%)",
    "ch4_category": "Methane category",
    "tddm_category": "TDDM category",
    "lmf_category": "LMF category",
    "lmf_category_rank": "Position on LMF category",
    "quartile": "LMF quartile",
    "quartile_rank": "LMF ranking"
}


# ============================================================
# TOOLTIP COLUMNS
# ============================================================

tooltip_columns = [
    "id",
    "id_lab",
    "tax_name",
    "functional_group",
    "subset",
    "n_replicates_nutrition",
    "dm_percentage",
    "ash_dm",
    "om_percentage",
    "pc_percentage_dm",
    "adf_percentage_dm",
    "ndf_percentage_dm",
    "n_replicates_gas",
    "ch4_percentage_in_gas_8h",
    "ch4_percentage_in_gas_24h",
    "methane_intensity",
    "tddm",
    "ch4_category",
    "tddm_category",
    "lmf_category",
    "lmf_category_rank",
    "quartile",
    "quartile_rank"
]


# ============================================================
# CHECK DBSCAN VARIABLES
# ============================================================

missing_dbscan_vars = [
    column
    for column in dbscan_vars
    if column not in df.columns
]

if missing_dbscan_vars:

    raise ValueError(
        "The following DBSCAN variables are missing from df: "
        f"{missing_dbscan_vars}"
    )


# ============================================================
# CLEAN FUNCTIONAL GROUP
# ============================================================

df["functional_group"] = (
    df["functional_group"]
    .astype("string")
    .str.strip()
)


# ============================================================
# VALID FUNCTIONAL GROUPS
# ============================================================

valid_functional_groups = [
    "Grasses",
    "Herbaceous_legumes",
    "Shrub_Trees"
]


# ============================================================
# FILTER DATA FOR DBSCAN
# ============================================================

df_dbscan = df[
    df["functional_group"].isin(
        valid_functional_groups
    )
].copy()


# ============================================================
# SPECIAL SAMPLE CATEGORIES
# ============================================================

# Default category = functional group

df_dbscan["point_category"] = (
    df_dbscan["functional_group"]
)


# ------------------------------------------------------------
# Benchmark samples
# ------------------------------------------------------------

df_dbscan.loc[
    df_dbscan["id"].isin(benchmark_ids),
    "point_category"
] = "Benchmark"


# ------------------------------------------------------------
# Star grass control
# ------------------------------------------------------------

df_dbscan.loc[
    df_dbscan["id"].isin(star_grass_control_ids),
    "point_category"
] = "Star grass control"


# ------------------------------------------------------------
# Target sample
# ------------------------------------------------------------

df_dbscan.loc[
    df_dbscan["id"].isin(target),
    "point_category"
] = "Target"


# ============================================================
# FUNCTIONAL GROUPS AVAILABLE FOR WIDGET
# ============================================================

functional_groups = sorted(
    df_dbscan["functional_group"]
    .dropna()
    .unique()
    .tolist()
)


# ============================================================
# FUNCTIONAL GROUP WIDGET
# ============================================================

dbscan_functional_widget = widgets.Dropdown(
    options=["All"] + functional_groups,
    value="All",
    description="Group:"
)


# ============================================================
# DBSCAN PARAMETERS
# ============================================================

# Initial values
initial_eps = 1.4
initial_min_samples = 5


# ============================================================
# EPS WIDGET
# ============================================================

dbscan_eps_widget = widgets.FloatSlider(
    value=initial_eps,
    min=0.1,
    max=5.0,
    step=0.1,
    description="eps:",
    continuous_update=False,
    readout_format=".1f",
    style={
        "description_width": "initial"
    },
    layout=widgets.Layout(
        width="500px"
    )
)


# ============================================================
# MIN SAMPLES WIDGET
# ============================================================

dbscan_min_samples_widget = widgets.IntSlider(
    value=initial_min_samples,
    min=2,
    max=30,
    step=1,
    description="min_samples:",
    continuous_update=False,
    style={
        "description_width": "initial"
    },
    layout=widgets.Layout(
        width="500px"
    )
)


# ============================================================
# DBSCAN FUNCTION
# ============================================================

def plot_dbscan(
    functional_group,
    eps,
    min_samples
):


    # ========================================================
    # FILTER DATA BY FUNCTIONAL GROUP
    # ========================================================

    if functional_group == "All":

        data = df_dbscan.copy()

    else:

        data = df_dbscan[
            df_dbscan["functional_group"]
            == functional_group
        ].copy()


    # ========================================================
    # CHECK NUMBER OF OBSERVATIONS
    # ========================================================

    if len(data) < min_samples:

        print(
            f"Not enough observations for "
            f"'{functional_group}'."
        )

        print(
            f"Found {len(data)} observations, "
            f"but min_samples = {min_samples}."
        )

        return


    # ========================================================
    # DBSCAN MATRIX
    # ========================================================

    X = data[dbscan_vars].copy()


    # ========================================================
    # REPLACE INFINITE VALUES WITH NaN
    # ========================================================

    X.replace(
        [np.inf, -np.inf],
        np.nan,
        inplace=True
    )


    # ========================================================
    # IMPUTE MISSING VALUES
    # ========================================================

    X = X.fillna(
        X.mean()
    )


    # ========================================================
    # CHECK REMAINING NaNs
    # ========================================================

    if X.isna().any().any():

        bad_columns = X.columns[
            X.isna().any()
        ].tolist()

        print(
            "These variables still contain missing values:"
        )

        print(
            bad_columns
        )

        return


    # ========================================================
    # CHECK ZERO VARIANCE
    # ========================================================

    zero_variance = [
        column
        for column in dbscan_vars
        if X[column].nunique() <= 1
    ]

    if zero_variance:

        print(
            "These variables have zero variance "
            "for this functional group:"
        )

        print(
            zero_variance
        )

        return


    # ========================================================
    # STANDARDIZATION
    # ========================================================

    scaler = StandardScaler()

    X_scaled = scaler.fit_transform(
        X
    )


    # ========================================================
    # DBSCAN
    # ========================================================

    dbscan = DBSCAN(
        eps=eps,
        min_samples=min_samples,
        metric="euclidean"
    )


    cluster_labels = dbscan.fit_predict(
        X_scaled
    )


    # ========================================================
    # SAVE CLUSTER LABELS
    # ========================================================

    data["cluster"] = cluster_labels


    # ========================================================
    # NUMBER OF CLUSTERS
    # ========================================================

    unique_clusters = set(
        cluster_labels
    )

    n_clusters = len(
        unique_clusters - {-1}
    )


    # ========================================================
    # NUMBER OF NOISE OBSERVATIONS
    # ========================================================

    n_noise = np.sum(
        cluster_labels == -1
    )


    # ========================================================
    # CLUSTER SUMMARY
    # ========================================================

    cluster_counts = (
        data["cluster"]
        .value_counts()
        .sort_index()
    )


    # ========================================================
    # PRINT SUMMARY
    # ========================================================

    print(
        "===================================================="
    )

    print(
        "DBSCAN SUMMARY"
    )

    print(
        "===================================================="
    )

    print(
        f"Functional group : {functional_group}"
    )

    print(
        f"Observations     : {len(data)}"
    )

    print(
        f"Variables        : {len(dbscan_vars)}"
    )

    print(
        f"eps              : {eps}"
    )

    print(
        f"min_samples      : {min_samples}"
    )

    print(
        f"Clusters found   : {n_clusters}"
    )

    print(
        f"Noise             : {n_noise}"
    )

    print(
        "----------------------------------------------------"
    )

    print(
        "Observations per cluster:"
    )

    print(
        cluster_counts
    )

    print(
        "===================================================="
    )


    # ========================================================
    # UMAP FOR VISUALIZATION ONLY
    # ========================================================

    if len(data) < 3:

        print(
            "Not enough observations for UMAP visualization."
        )

        return


    # --------------------------------------------------------
    # Adjust n_neighbors
    # --------------------------------------------------------

    effective_n_neighbors = min(
        15,
        len(data) - 1
    )


    # --------------------------------------------------------
    # UMAP
    # --------------------------------------------------------

    reducer = umap.UMAP(
        n_components=2,
        n_neighbors=effective_n_neighbors,
        min_dist=0.1,
        metric="euclidean",
        random_state=42
    )


    embedding = reducer.fit_transform(
        X_scaled
    )


    # ========================================================
    # UMAP COORDINATES
    # ========================================================

    data["UMAP1"] = embedding[:, 0]

    data["UMAP2"] = embedding[:, 1]


    # ========================================================
    # CLUSTER LABEL
    # ========================================================

    data["cluster_label"] = (
        data["cluster"]
        .astype(str)
    )


    data.loc[
        data["cluster"] == -1,
        "cluster_label"
    ] = "Noise"


    # ========================================================
    # TOOLTIP COLUMNS
    # ========================================================

    available_tooltip_columns = [
        column
        for column in tooltip_columns
        if column in data.columns
    ]


    # ========================================================
    # TOOLTIP
    # ========================================================

    tooltip = [

        alt.Tooltip(
            column,
            title=column_labels.get(
                column,
                column.replace(
                    "_",
                    " "
                ).title()
            )
        )

        for column in available_tooltip_columns

    ]


    # --------------------------------------------------------
    # Add cluster
    # --------------------------------------------------------

    tooltip.append(
        alt.Tooltip(
            "cluster:N",
            title="DBSCAN cluster"
        )
    )


    # --------------------------------------------------------
    # Add UMAP coordinates
    # --------------------------------------------------------

    tooltip += [

        alt.Tooltip(
            "UMAP1:Q",
            title="UMAP1",
            format=".3f"
        ),

        alt.Tooltip(
            "UMAP2:Q",
            title="UMAP2",
            format=".3f"
        )

    ]


    # ========================================================
    # DBSCAN POINTS
    # ========================================================

    points = (

        alt.Chart(
            data
        )

        .mark_circle(
            size=80
        )

        .encode(

            # ------------------------------------------------
            # X axis
            # ------------------------------------------------

            x=alt.X(
                "UMAP1:Q",
                title="UMAP 1"
            ),


            # ------------------------------------------------
            # Y axis
            # ------------------------------------------------

            y=alt.Y(
                "UMAP2:Q",
                title="UMAP 2"
            ),


            # ------------------------------------------------
            # COLOR BY CLUSTER
            # ------------------------------------------------

            color=alt.Color(
                "cluster_label:N",
                title="DBSCAN cluster"
            ),


            # ------------------------------------------------
            # TOOLTIP
            # ------------------------------------------------

            tooltip=tooltip

        )

    )


    # ========================================================
    # FINAL PLOT
    # ========================================================

    final_plot = (

        points

        .properties(

            width=850,

            height=600,

            title=(
                f"DBSCAN — "
                f"{functional_group} "
                f"(n = {len(data)}, "
                f"clusters = {n_clusters}, "
                f"noise = {n_noise})"
            )

        )

        .configure_axis(

            labelFontSize=14,

            titleFontSize=18

        )

        .configure_legend(

            titleFontSize=16,

            labelFontSize=14

        )

        .configure_title(

            fontSize=20

        )

        .interactive()

    )


    # ========================================================
    # DISPLAY
    # ========================================================

    display(
        final_plot
    )





In [53]:
# ============================================================
# Requirements
# ============================================================
# pip install altair vl-convert-python pandas numpy scikit-learn umap-learn
#
# IMPORTANT — read this before running:
# A static HTML file cannot execute Python, so it cannot run a new
# DBSCAN fit every time you move a slider. Instead, we precompute
# DBSCAN for a GRID of (functional_group, eps, min_samples) values,
# and the HTML lets you pick among those precomputed combinations
# using dropdowns instead of free-moving sliders.
#
# Adjust EPS_GRID / MIN_SAMPLES_GRID below to trade off resolution
# vs. file size: more values = finer control, but a bigger HTML file.

import numpy as np
import pandas as pd
import altair as alt
import umap
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN

# Disable Altair's default 5000-row safety limit: we deliberately
# embed a precomputed grid (group x eps x min_samples) that can
# exceed this, and that's expected for this static HTML export.
alt.data_transformers.disable_max_rows()

# ============================================================
# DBSCAN variables
# ============================================================

dbscan_vars = [
    "dm_percentage",
    "ash_dm",
    "om_percentage",
    "pc_percentage_dm",
    "adf_percentage_dm",
    "ndf_percentage_dm",
    "ch4_percentage_in_gas_8h",
    "ch4_percentage_in_gas_24h",
    "methane_intensity",
    "tddm"
]

# ============================================================
# Column labels
# ============================================================

column_labels = {
    "id": "Accession identifier",
    "id_lab": "Laboratory sample identifier",
    "tax_name": "Taxonomic name",
    "functional_group": "Functional group",
    "subset": "Experimental subset",
    "n_replicates_nutrition": "No. replicates (Nutrition)",
    "dm_percentage": "Dry matter (%)",
    "ash_dm": "Ash (% DM)",
    "om_percentage": "Organic matter (% DM)",
    "pc_percentage_dm": "Crude protein (% DM)",
    "adf_percentage_dm": "ADF (% DM)",
    "ndf_percentage_dm": "NDF (% DM)",
    "n_replicates_gas": "No. replicates (Gas)",
    "ch4_percentage_in_gas_8h": "Methane in Gas (8 h, %)",
    "ch4_percentage_in_gas_24h": "Methane in Gas (24 h, %)",
    "methane_intensity": "Methane intensity",
    "tddm": "TDDM (%)",
    "ch4_category": "Methane category",
    "tddm_category": "TDDM category",
    "lmf_category": "LMF category",
    "lmf_category_rank": "Position on LMF category",
    "quartile": "LMF quartile",
    "quartile_rank": "LMF ranking"
}

tooltip_columns = [
    "id", "id_lab", "tax_name", "functional_group",
    "subset", "n_replicates_nutrition", "dm_percentage", "ash_dm",
    "om_percentage", "pc_percentage_dm", "adf_percentage_dm",
    "ndf_percentage_dm", "n_replicates_gas", "ch4_percentage_in_gas_8h",
    "ch4_percentage_in_gas_24h", "methane_intensity", "tddm",
    "ch4_category", "tddm_category", "lmf_category",
    "lmf_category_rank", "quartile", "quartile_rank"
]

# ============================================================
# Precomputed grid — tune these to control file size/resolution
# ============================================================

EPS_GRID = [round(v, 1) for v in np.arange(0.4, 2.01, 0.2)]     # e.g. 0.4, 0.6, ..., 2.0
MIN_SAMPLES_GRID = [3, 5, 8, 12, 16, 20]

# ============================================================
# NOTE: this assumes `df_dbscan` has already been built exactly as
# in the original script (filtered, with point_category assigned, etc.)
# ============================================================

missing_dbscan_vars = [c for c in dbscan_vars if c not in df_dbscan.columns]
if missing_dbscan_vars:
    raise ValueError(f"The following DBSCAN variables are missing from df_dbscan: {missing_dbscan_vars}")

functional_groups = sorted(df_dbscan["functional_group"].dropna().unique().tolist())
groups_to_run = ["All"] + functional_groups


def prepare_group_matrix(data):
    """Clean + scale the feature matrix for one functional group subset."""

    X = data[dbscan_vars].copy()
    X.replace([np.inf, -np.inf], np.nan, inplace=True)
    X = X.fillna(X.mean())

    if X.isna().any().any():
        bad = X.columns[X.isna().any()].tolist()
        print(f"These variables still contain missing values: {bad}")
        return None, None

    zero_var = [c for c in dbscan_vars if X[c].nunique() <= 1]
    if zero_var:
        print(f"These variables have zero variance: {zero_var}")
        return None, None

    X_scaled = StandardScaler().fit_transform(X)
    return X, X_scaled


# ============================================================
# Pre-compute UMAP (once per group — layout only depends on the
# group's data, not on eps/min_samples) and DBSCAN (once per
# group x eps x min_samples combination in the grid)
# ============================================================

all_rows = []

for group in groups_to_run:

    data = df_dbscan.copy() if group == "All" else df_dbscan[df_dbscan["functional_group"] == group].copy()

    if len(data) < 3:
        print(f"Not enough observations for '{group}'. Found {len(data)} samples.")
        continue

    X, X_scaled = prepare_group_matrix(data)
    if X is None:
        continue

    # --------------------------------------------------------
    # UMAP layout (fixed for this group, reused across the grid)
    # --------------------------------------------------------

    effective_n_neighbors = min(15, len(data) - 1)

    reducer = umap.UMAP(
        n_components=2,
        n_neighbors=effective_n_neighbors,
        min_dist=0.1,
        metric="euclidean",
        random_state=42
    )
    embedding = reducer.fit_transform(X_scaled)

    base = data.copy()
    base["UMAP1"] = embedding[:, 0]
    base["UMAP2"] = embedding[:, 1]

    available_tooltip_cols = [c for c in tooltip_columns if c in base.columns]

    # --------------------------------------------------------
    # DBSCAN for every (eps, min_samples) combination
    # --------------------------------------------------------

    for eps in EPS_GRID:
        for min_samples in MIN_SAMPLES_GRID:

            if len(data) < min_samples:
                continue

            dbscan = DBSCAN(eps=eps, min_samples=min_samples, metric="euclidean")
            cluster_labels = dbscan.fit_predict(X_scaled)

            combo_df = base[available_tooltip_cols + ["UMAP1", "UMAP2"]].copy()
            combo_df["cluster"] = cluster_labels
            # Display labels start at 1 instead of DBSCAN's native 0-based
            # numbering (the raw "cluster" column, e.g. for the tooltip,
            # keeps DBSCAN's original 0 / 1 / 2... / -1 values).
            combo_df["cluster_label"] = (combo_df["cluster"] + 1).astype(str)
            combo_df.loc[combo_df["cluster"] == -1, "cluster_label"] = "Noise"

            combo_df["umap_run"] = group
            combo_df["eps_label"] = f"{eps:.1f}"
            combo_df["min_samples"] = min_samples

            all_rows.append(combo_df)

points_all = pd.concat(all_rows, ignore_index=True)

# ============================================================
# Interactive controls: group / eps / min_samples dropdowns
# ============================================================
# Plain alt.param (not selection_point) so each value can be
# referenced directly by name inside Vega expressions.

group_param = alt.param(
    name="sel_group",
    value="All",
    bind=alt.binding_select(options=groups_to_run, name="Functional group: ")
)

eps_options = sorted(points_all["eps_label"].unique().tolist(), key=float)
eps_param = alt.param(
    name="sel_eps",
    value="1.4",
    bind=alt.binding_select(options=eps_options, name="eps: ")
)

min_samples_options = sorted(points_all["min_samples"].unique().tolist())
min_samples_param = alt.param(
    name="sel_min_samples",
    value=5,
    bind=alt.binding_select(options=min_samples_options, name="min_samples: ")
)

combo_filter = (
    "datum.umap_run == sel_group && "
    "datum.eps_label == sel_eps && "
    "datum.min_samples == sel_min_samples"
)

# ------------------------------------------------------------
# Text search box: highlight by id or id_lab, comma-separated,
# partial (substring) match, case-insensitive
# ------------------------------------------------------------

search_param = alt.param(
    name="search_id",
    value="",
    bind=alt.binding(input="text", name="Search ID or Lab ID (comma-separated): ")
)

search_pattern_expr = (
    "replace(replace(search_id, /\\s+/g, ''), /,/g, '|')"
)

is_match_expr = (
    f"search_id !== '' && ("
    f"test(regexp({search_pattern_expr}, 'i'), toString(datum.id)) || "
    f"test(regexp({search_pattern_expr}, 'i'), toString(datum.id_lab))"
    f")"
)

tooltip = [
    alt.Tooltip(col, title=column_labels.get(col, col.replace("_", " ").title()))
    for col in tooltip_columns
    if col in points_all.columns
] + [
    alt.Tooltip("cluster:N", title="DBSCAN cluster"),
    alt.Tooltip("UMAP1:Q", title="UMAP1", format=".3f"),
    alt.Tooltip("UMAP2:Q", title="UMAP2", format=".3f"),
]

# ------------------------------------------------------------
# Title row
# ------------------------------------------------------------

title_chart = (
    alt.Chart(points_all)
    .transform_filter(combo_filter)
    .transform_aggregate(n="count()")
    .transform_calculate(label="'DBSCAN & UPGMA —  ' + sel_group + ' (n = ' + datum.n + ')'")
    .mark_text(align="center", fontSize=20, dy=0)
    .encode(text="label:N")
    .properties(width=850, height=30)
)

# ------------------------------------------------------------
# DBSCAN points (colored by cluster)
# ------------------------------------------------------------

base_layer = (
    alt.Chart(points_all)
    .mark_circle(size=80)
    .transform_filter(combo_filter)
    .transform_filter(f"!({is_match_expr})")
    .transform_calculate(
        cluster_sort_key="datum.cluster_label === 'Noise' ? -9999 : toNumber(datum.cluster_label)"
    )
    .encode(
        x=alt.X("UMAP1:Q", title="UMAP 1"),
        y=alt.Y("UMAP2:Q", title="UMAP 2"),
        color=alt.Color(
            "cluster_label:N",
            title="DBSCAN cluster",
            sort=alt.EncodingSortField(field="cluster_sort_key", op="min", order="ascending")
        ),
        tooltip=tooltip
    )
)

highlight_layer = (
    alt.Chart(points_all)
    .mark_point(filled=True, size=80, stroke="black", strokeWidth=1.5)
    .transform_filter(combo_filter)
    .transform_filter(is_match_expr)
    .encode(
        x=alt.X("UMAP1:Q"),
        y=alt.Y("UMAP2:Q"),
        # Only matched rows are ever present in this layer's data, so the
        # shape scale's domain — and therefore the legend — automatically
        # lists exactly the searched accession(s), and only those.
        shape=alt.Shape(
            "id:N",
            legend=alt.Legend(title="Highlighted accession(s)", symbolSize=80, titleLimit=0)
        ),
        color=alt.value("red"),
        tooltip=tooltip
    )
)

points_chart = base_layer + highlight_layer

biplot = points_chart.properties(width=850, height=600)

# ============================================================
# Final layout — controls above, then title, plot, and table
# ============================================================

final_plot = (
    alt.vconcat(
        title_chart,
        biplot
    )
    .add_params(group_param, eps_param, min_samples_param, search_param)
    .configure_axis(labelFontSize=14, titleFontSize=18)
    .configure_legend(titleFontSize=16, labelFontSize=14)
    .configure_view(strokeWidth=0)
    .interactive()
)

# ============================================================
# Save as interactive HTML with controls forced above the chart
# ============================================================

chart_spec = final_plot.to_json(indent=None)

html_template = f"""
<!DOCTYPE html>
<html>
<head>
  <meta charset="utf-8" />
  <title>DBSCAN Plot</title>
  <script src="https://cdn.jsdelivr.net/npm/vega@5"></script>
  <script src="https://cdn.jsdelivr.net/npm/vega-lite@5"></script>
  <script src="https://cdn.jsdelivr.net/npm/vega-embed@6"></script>
  <script src="https://cdn.jsdelivr.net/npm/utif@3.1.0/UTIF.js"></script>
  <script src="https://cdnjs.cloudflare.com/ajax/libs/jspdf/2.5.1/jspdf.umd.min.js"></script>
  <style>
    body {{
      font-family: sans-serif;
      display: flex;
      flex-direction: column;
      align-items: center;
    }}
    #controls {{
      margin: 20px 0;
      font-size: 16px;
    }}
    #download-buttons {{
      margin: 20px 0 10px 0;
      display: flex;
      align-items: center;
      gap: 10px;
    }}
    #download-buttons select,
    #download-buttons button {{
      padding: 8px 16px;
      font-size: 14px;
      border: 1px solid #ccc;
      border-radius: 6px;
      background: #f7f7f9;
      cursor: pointer;
    }}
    #download-buttons button:hover {{
      background: #e9e9ee;
    }}
  </style>
</head>
<body>
  <div id="controls"></div>
  <div id="vis"></div>
  <div id="download-buttons">
    <select id="format-select">
      <option value="png">PNG</option>
      <option value="jpg">JPG</option>
      <option value="tiff">TIFF</option>
      <option value="pdf">PDF</option>
    </select>
    <button id="btn-download">Download</button>
  </div>

  <script type="text/javascript">
    const spec = {chart_spec};

    // Vega renders at 96 CSS pixels per inch by default. Scaling the
    // canvas by (300 / 96) makes the exported pixel dimensions match
    // a 300 dpi print at the chart's original physical size, for
    // every format below (PNG, JPG, TIFF, and PDF alike).
    const DPI = 300;
    const DPI_SCALE = DPI / 96;

    vegaEmbed("#vis", spec, {{
      actions: false,
      bindingsElement: "#controls",
      renderer: "svg"
    }}).then(function(result) {{

      const view = result.view;

      function triggerDownload(url, filename) {{
        const link = document.createElement("a");
        link.href = url;
        link.download = filename;
        document.body.appendChild(link);
        link.click();
        document.body.removeChild(link);
      }}

      function downloadAs(format) {{
        view.toCanvas(DPI_SCALE).then(function(canvas) {{

          if (format === "png") {{
            triggerDownload(canvas.toDataURL("image/png"), "dbscan_plot.png");

          }} else if (format === "jpg") {{
            triggerDownload(canvas.toDataURL("image/jpeg", 0.95), "dbscan_plot.jpg");

          }} else if (format === "tiff") {{
            const ctx = canvas.getContext("2d");
            const imageData = ctx.getImageData(0, 0, canvas.width, canvas.height);
            const tiffBuffer = UTIF.encodeImage(imageData.data, canvas.width, canvas.height);
            const blob = new Blob([tiffBuffer], {{ type: "image/tiff" }});
            const url = URL.createObjectURL(blob);
            triggerDownload(url, "dbscan_plot.tiff");

          }} else if (format === "pdf") {{
            // Page size (in inches) = pixel dimensions / DPI, so the
            // embedded image lands at exactly 300 dpi on the page.
            const widthIn = canvas.width / DPI;
            const heightIn = canvas.height / DPI;
            const {{ jsPDF }} = window.jspdf;
            const pdf = new jsPDF({{
              orientation: widthIn >= heightIn ? "landscape" : "portrait",
              unit: "in",
              format: [widthIn, heightIn]
            }});
            pdf.addImage(canvas.toDataURL("image/png"), "PNG", 0, 0, widthIn, heightIn);
            pdf.save("dbscan_plot.pdf");
          }}

        }}).catch(console.error);
      }}

      document.getElementById("btn-download").addEventListener("click", function() {{
        const format = document.getElementById("format-select").value;
        downloadAs(format);
      }});

      // Explicitly hide the "Highlighted accession(s)" legend while the
      // search box is empty (on top of Vega-Lite's own behavior of not
      // populating it when the shape scale's domain has no data).
      function toggleHighlightedAccessionsLegend(searchValue) {{
        const container = view.container();
        if (!container) return;

        const isActive = !!(searchValue && searchValue.trim() !== "");

        // Vega tags each legend group with aria-roledescription="legend"
        // for accessibility — this is more stable across versions than
        // relying on internal CSS class names.
        const legendGroups = container.querySelectorAll('[aria-roledescription="legend"]');

        legendGroups.forEach(function(g) {{
          const texts = g.querySelectorAll("text");
          let isTargetLegend = false;
          texts.forEach(function(t) {{
            if (t.textContent.trim().indexOf("Highlighted") === 0) {{
              isTargetLegend = true;
            }}
          }});
          if (isTargetLegend) {{
            g.style.display = isActive ? "" : "none";
          }}
        }});
      }}

      toggleHighlightedAccessionsLegend(view.signal("search_id"));
      view.addSignalListener("search_id", function(name, value) {{
        // Small delay: the signal fires slightly before Vega finishes
        // redrawing the SVG (e.g. adding/removing the legend group).
        setTimeout(function() {{ toggleHighlightedAccessionsLegend(value); }}, 50);
      }});

    }}).catch(console.error);
  </script>
</body>
</html>
"""

with open("dbscan.html", "w", encoding="utf-8") as f:
    f.write(html_template)

print("Done: dbscan.html")
print(f"Precomputed combinations: {len(groups_to_run)} groups x {len(EPS_GRID)} eps x {len(MIN_SAMPLES_GRID)} min_samples")
print(f"Total precomputed rows: {len(points_all)}")

/usr/local/lib/python3.13/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/usr/local/lib/python3.13/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/usr/local/lib/python3.13/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/usr/local/lib/python3.13/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Done: dbscan.html
Precomputed combinations: 4 groups x 9 eps x 6 min_samples
Total precomputed rows: 74088
